# 02 - Modelos lineares generalizados (Partes 5 a 10)

Ajuste de todas as especificações candidatas à base de 5.570 municípios. Este notebook **compara e não escolhe**:
a escolha do modelo final é feita por validação cruzada no notebook 03, com as mesmas definições de `src/especificacoes.py`.

## Parte 5 - Formulação

**Resposta.** Para o município *i*: Yᵢ = votos de Lula, nᵢ = votos de Lula + votos de Flávio Bolsonaro (denominador do 2º turno),
Yᵢ ~ Bin(nᵢ, πᵢ), g(πᵢ) = xᵢᵀβ. Um único modelo dá os dois candidatos, pois o share de Flávio é 1 − πᵢ.

**Por que o binomial agrupado.** Se cada eleitor do município *i* vota em Lula com probabilidade πᵢ, independentemente dos demais, a verossimilhança
dos nᵢ votos individuais é ∏ πᵢ^yᵢ (1−πᵢ)^(nᵢ−yᵢ). Ela é igual, a menos da constante binomial C(nᵢ, yᵢ), à do binomial agrupado em (yᵢ, nᵢ):
a regressão logística de voto individual com covariáveis **municipais** (todos os eleitores do município compartilham xᵢ) é exatamente o GLM binomial
agrupado. Os pesos nᵢ entram naturalmente: São Paulo pesa milhões de votos, uma cidade pequena pesa centenas.

**Falácia ecológica.** Os coeficientes descrevem como o *share municipal* varia com características *municipais*. Não dizem como um indivíduo evangélico
ou beneficiário do Bolsa Família vota. Interpretamos β como associações entre municípios e usamos o modelo para prever agregados, não para inferir comportamento individual.

**Famílias excluídas.** Gamma e normal inversa têm suporte em (0, ∞) e a Gaussiana em ℝ; a resposta é uma proporção em (0, 1) vinda de contagens com limite superior nᵢ.
Essas famílias não respeitam o suporte nem a relação média-variância πᵢ(1−πᵢ)/nᵢ. Ficam apenas a Gaussiana sobre o logit empírico (linha de base, Parte 7) e as famílias de proporção (binomial, quase-binomial, beta-binomial, beta).

### Teoria - Componentes de um MLG e família exponencial

Um MLG tem três componentes: (i) **componente aleatório**, $Y_i$ com distribuição na família exponencial; (ii) **componente sistemático**, o preditor linear $\eta_i = x_i^\top\beta$; (iii) **função de ligação** $g$, com $g(\mu_i)=\eta_i$.

Na família exponencial de dispersão,
$$f(y;\theta,\phi)=\exp\left\{\frac{y\theta-b(\theta)}{a(\phi)}+c(y,\phi)\right\},\qquad E(Y)=b'(\theta),\quad \mathrm{Var}(Y)=a(\phi)\,b''(\theta)=a(\phi)\,V(\mu).$$

Para a proporção $Y_i/n_i$ com $Y_i\sim\mathrm{Bin}(n_i,\pi_i)$: $\theta=\log\frac{\pi}{1-\pi}$, $b(\theta)=\log(1+e^\theta)$, $a(\phi)=1/n_i$ e **função de variância** $V(\mu)=\mu(1-\mu)$, de modo que $\mathrm{Var}(Y_i/n_i)=\pi_i(1-\pi_i)/n_i$. A variância depende da média: não há parâmetro de variância livre, e é isso que a sobredispersão (Parte 7) viola.

Com a ligação canônica (logit), as equações de escore são $U(\beta)=\sum_i x_i\,(y_i-n_i\pi_i)=0$: o ajuste iguala a soma observada e a ajustada de votos em cada direção de $x$ (com dummies de região, reproduz os totais regionais).

In [1]:
import sys, json, warnings, time
from concurrent.futures import ProcessPoolExecutor
from dataclasses import replace
from functools import partial
from multiprocessing import get_context
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, esda
from scipy import stats
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif
from libpysal.weights import Queen, KNN
from libpysal.weights.util import attach_islands
from plotly.subplots import make_subplots
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "notebooks"))
from src import especificacoes as E, modelos as M, selecao as SEL, espacial as S, pressupostos as PR
from src.especificacoes import Candidato, Termos
import graficos_diagnostico as G
pd.options.display.float_format = "{:,.4f}".format
T0 = time.time()
TAB, FIG = ROOT / "results/tables", ROOT / "results/figures"
TAB.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
df = pd.read_parquet(ROOT / "data/processed/municipios.parquet")
mapa = E.MAPA_2026
y, n = df[mapa["y"]].to_numpy(float), df[mapa["n"]].to_numpy(float)
N = len(df)
print(N, "municípios;", int(y.sum()), "votos Lula de", int(n.sum()), f"= {y.sum()/n.sum():.4%}")
AJ, LINHAS = {}, {}
def ajustar(nome):
    c = E.CANDIDATOS[nome]; a = c.ajustar(df, mapa); AJ[nome] = a
    LINHAS[nome] = E.resumo_ajuste(c, a, df, mapa)
    E.tabela_coeficientes(a).to_csv(TAB / f"coef_{nome}.csv", index_label="termo")
    return a

5570 municípios; 64055133 votos Lula de 123707324 = 51.7796%


**Totais regionais exatos dentro da amostra.** Com a ligação canônica (logit) e dummies de região, as equações de escore do GLM binomial implicam
Σᵢ∈R nᵢ π̂ᵢ = Σᵢ∈R yᵢ para cada região R (as dummies e a constante geram a combinação). O ajuste reproduz os totais regionais *por construção*.
Isso vale só dentro da amostra e **não mede capacidade de previsão regional**: as estimativas regionais devem vir de previsão fora da amostra (notebook 03).

In [2]:
a = ajustar("bin_logit")
pi = E.CANDIDATOS["bin_logit"].prever(a, df, mapa)
reg = pd.DataFrame({"regiao": df[mapa["regiao"]], "observado": y, "ajustado": pi * n}).groupby("regiao").sum()
reg["diferenca"] = reg.ajustado - reg.observado
print(reg.round(4).to_string())
print("máx |diferença| (votos):", reg.diferenca.abs().max(), "| total observado:", reg.observado.sum())

                   observado        ajustado  diferenca
regiao                                                 
Centro-Oeste  3,732,974.0000  3,732,974.0000    -0.0000
Nordeste     24,450,432.0000 24,450,432.0000    -0.0000
Norte         5,024,626.0000  5,024,626.0000    -0.0000
Sudeste      24,037,731.0000 24,037,731.0000    -0.0000
Sul           6,809,370.0000  6,809,370.0000    -0.0000
máx |diferença| (votos): 8.568167686462402e-08 | total observado: 64055133.0


## Parte 6 - Binomial e ligações

**Multicolinearidade.** Calculamos o VIF das covariáveis padronizadas antes de ajustar. A regra do projeto é remover uma covariável de um par só se VIF > 10.

In [3]:
pad = E.Padronizador(mapa).fit(df); X = E.montar_X(df, mapa, pad)
VIF = pd.Series({c: vif(X.values, i) for i, c in enumerate(X.columns) if c != "const"}).sort_values(ascending=False)
VIF.to_frame("VIF").to_csv(TAB / "vif_modelo.csv", index_label="termo")
print(VIF.head(8).round(1).to_string()); print("máx VIF:", round(VIF.max(), 1), "| colunas:", X.shape[1] - 1)

pct_16_24           7.5000
cobertura_pbf       6.1000
lag                 5.8000
pct_60_mais         5.3000
reg_Nordeste        5.3000
pct_pretos_pardos   4.7000
log_dens            3.5000
pct_mulheres        3.5000
máx VIF: 7.5 | colunas: 24


**Decisão:** nenhum VIF passa de 10 (o maior é `pct_16_24`, 7,6), então **nenhuma covariável é removida** por colinearidade. A categoria `bloco_prefeito_2024`
tem só 49 municípios 'centro' e 16 'sem_prefeito'; **fundimos os dois em 'outro'** (65 municípios) para a dummy ter estimativa estável. Referências: bloco 'direita' e região 'Sudeste'.
As contínuas entram em z-score, com média e desvio calculados no treino.

**Ligações binomiais.** Comparamos logit, probit, cloglog, loglog e cauchit com o mesmo preditor linear. Com a mesma resposta e o mesmo *k*, a log-verossimilhança (e portanto AIC/BIC) é diretamente comparável.
Reportamos deviance, deviance/gl e φ de Pearson (definições nas células de Teoria desta Parte e da Parte 7).

### Teoria - Ligação canônica, IRLS, deviance e critérios

**Ligação canônica** é aquela em que $g(\mu)=\theta$; para o binomial, o logit. Ela dá estatísticas suficientes $X^\top y$ e log-verossimilhança côncava em $\beta$ (solução única, convergência estável). As demais ligações (probit, cloglog...) são válidas, mas perdem essas propriedades.

**IRLS / escore de Fisher.** A cada iteração, com $\eta=X\beta$, $\mu=g^{-1}(\eta)$,
$$z=\eta+(y-\mu)\,g'(\mu),\qquad W=\frac{1}{V(\mu)\,g'(\mu)^2},\qquad \beta\leftarrow(X^\top WX)^{-1}X^\top Wz,$$
ou seja, uma regressão de mínimos quadrados ponderados sobre uma resposta de trabalho $z$. No logit, $W=\mathrm{diag}(n_i\pi_i(1-\pi_i))$; $\widehat{\mathrm{Var}}(\hat\beta)=\phi\,(X^\top WX)^{-1}$.

**Deviance.** $D=2\sum_i[\ell(\text{saturado})-\ell(\text{ajustado})]$. Para o binomial,
$$D=2\sum_i\left[y_i\log\frac{y_i}{n_i\hat\pi_i}+(n_i-y_i)\log\frac{n_i-y_i}{n_i(1-\hat\pi_i)}\right],$$
e a deviance escalada é $D/\phi$. Se o modelo é correto e $\phi=1$, $D\approx\chi^2_{N-p}$, logo $D/\mathrm{gl}\approx1$; valores ≫ 1 indicam sobredispersão (aqui, ≈ 20).

**Critérios.** $\mathrm{AIC}=-2\ell+2k$, $\mathrm{BIC}=-2\ell+k\log N$ e, para o quase-binomial, $\mathrm{QAIC}=-2\ell/\hat\phi+2k$. Só valem para modelos com verossimilhança completa e mesma resposta; o QAIC usa o $\hat\phi$ do modelo mais geral.

In [4]:
for nome in ["bin_logit", "bin_probit", "bin_cloglog", "bin_loglog", "bin_cauchit"]:
    ajustar(nome)
cols = ["modelo", "ligacao", "k", "loglik", "aic", "bic", "deviance", "deviance_gl", "phi_pearson", "mae_pond_pp"]
t6 = pd.DataFrame([LINHAS[k] for k in ["bin_logit", "bin_probit", "bin_cloglog", "bin_loglog", "bin_cauchit"]])[cols]
print(t6.to_string(index=False))

     modelo ligacao  k        loglik          aic          bic     deviance  deviance_gl  phi_pearson  mae_pond_pp
  bin_logit   logit 25  -86,271.6466 172,593.2932 172,758.9220 120,618.7959      21.7527      21.7714       1.0630
 bin_probit  probit 25  -88,115.8299 176,281.6598 176,447.2886 124,307.1625      22.4179      22.4614       1.0747
bin_cloglog cloglog 25 -126,225.1415 252,500.2830 252,665.9118 200,525.7857      36.1634      36.5969       1.3538
 bin_loglog  loglog 25 -106,944.7683 213,939.5365 214,105.1653 161,965.0392      29.2092      29.1751       1.2018
bin_cauchit cauchit 25 -106,674.7675 213,399.5349 213,565.1637 161,425.0376      29.1118      28.7905       1.2096


**Leitura.** O logit tem o menor AIC/BIC entre as cinco ligações, de longe. Mas a deviance/gl é da ordem de 20 e o φ de Pearson é ≈ 20: com nᵢ de milhares de votos,
a variância binomial é muito menor que a variação real entre municípios. **Os erros-padrão do binomial puro são grosseiramente pequenos** e os testes não valem; a Parte 7 trata disso.
Escolher a ligação por AIC nesta base ainda é válido para ordenar os links, pois todos sofrem da mesma sobredispersão.

## Parte 7 - Sobredispersão

Quatro tratamentos, mais duas linhas de base:

1. **Quase-binomial (logit)**: mesmos coeficientes do binomial; φ estimado por Pearson. Não é uma verossimilhança, então não há AIC/BIC (reportamos QAIC).
2. **Beta-binomial (logit)** (`modelos.BetaBinomial`): πᵢ ~ Beta com precisão θ (definição na Teoria abaixo). É uma verossimilhança, logo tem AIC/BIC e é a que mais fielmente modela a dispersão extra.
3. **Regressão beta** (`BetaModel`, **sem pesos**): modela a proporção diretamente. Um município de 1.000 eleitores conta como São Paulo, por isso ela não é o objetivo (erro ponderado por votos); fica como comparação.
4. **OLS sobre o logit empírico** log((y+½)/(n−y+½)), sem pesos, e **WLS** com pesos nᵢp̂ᵢ(1−p̂ᵢ) (inverso da variância delta do logit): linhas de base simples.

### Teoria - Sobredispersão

Há sobredispersão quando $\mathrm{Var}(Y_i)>n_i\pi_i(1-\pi_i)$. O binomial supõe $n_i$ ensaios independentes com a mesma probabilidade; duas causas o violam:

1. **Correlação entre as réplicas binárias.** Eleitores do mesmo município se influenciam (família, igreja, lideranças locais), então os votos não são independentes.
2. **Variação da probabilidade dentro do grupo.** O binomial supõe a mesma probabilidade para todo eleitor do município; a probabilidade real varia entre bairros e grupos.

Com correlação intraclasse $\rho$ entre os votos,
$$\mathrm{Var}(Y_i)=n_i\pi_i(1-\pi_i)\,[1+(n_i-1)\rho].$$
Com $n_i$ na casa dos milhares, até um $\rho$ minúsculo infla muito a variância (o fator $1+(n_i-1)\rho$ passa de 20 com $\rho\approx 0{,}01$ e $n_i=2000$), o que explica $\hat\phi\approx21{,}6$ no modelo completo. **Consequência:** o binomial subestima a variância, os erros-padrão ficam pequenos demais e os testes, liberais.

**Quase-verossimilhança.** Especificam-se só a média e a variância, $\mathrm{Var}(Y_i)=\phi\,n_i\pi_i(1-\pi_i)$, com $\hat\phi=X^2/(N-p)$, $X^2=\sum_i\frac{(y_i-n_i\hat\pi_i)^2}{n_i\hat\pi_i(1-\hat\pi_i)}$. As equações de escore não mudam, então $\hat\beta$ é o do binomial e $\mathrm{EP}_{quasi}=\sqrt{\hat\phi}\,\mathrm{EP}_{bin}$. Não há verossimilhança, logo não há AIC/BIC nem razão de verossimilhanças genuína.

**Beta-binomial.** $\pi_i^*\sim\mathrm{Beta}$ com média $\pi_i$ e precisão $\theta$ (equivalente a $\rho=1/(1+\theta)$), e $Y_i\mid\pi_i^*\sim\mathrm{Bin}(n_i,\pi_i^*)$. Marginalmente, $E(Y_i)=n_i\pi_i$ (a mesma média do binomial) e $\mathrm{Var}(Y_i)=n_i\pi_i(1-\pi_i)[1+(n_i-1)\rho]$, que é exatamente a forma da variância acima. Como $\pi_i$ depende das covariáveis pela ligação ($g(\pi_i)=x_i^\top\beta$) e a verossimilhança é completa, AIC, BIC e razão de verossimilhanças são válidos, ao contrário do quase-binomial.

In [5]:
for nome in ["quasi_logit", "bb_logit", "betareg", "ols_elogit", "wls_elogit"]:
    ajustar(nome)
bb = AJ["bb_logit"].modelo
print(f"quase-binomial: phi = {AJ['quasi_logit'].extra['phi']:.2f}  (erros-padrão x {np.sqrt(AJ['quasi_logit'].extra['phi']):.2f})")
print(f"beta-binomial: theta = {M.theta_de(bb):.1f}, rho = {M.icc(bb):.5f}  (loglik {bb.llf:,.1f})")
lado = pd.concat({"binomial": E.tabela_coeficientes(AJ["bin_logit"])[["coef", "ep"]],
                  "quase-binomial": E.tabela_coeficientes(AJ["quasi_logit"])[["coef", "ep"]],
                  "beta-binomial": E.tabela_coeficientes(AJ["bb_logit"])[["coef", "ep"]]}, axis=1)
lado.to_csv(TAB / "coef_binomial_quasi_bb.csv", index_label="termo")
print(lado.round(3).to_string())

quase-binomial: phi = 21.77  (erros-padrão x 4.67)
beta-binomial: theta = 623.6, rho = 0.00160  (loglik -36,327.7)
                  binomial        quase-binomial        beta-binomial       
                      coef     ep           coef     ep          coef     ep
const               0.1910 0.0010         0.1910 0.0030        0.1980 0.0030
lag                 0.7770 0.0010         0.7770 0.0030        0.7750 0.0030
nota_dep            0.0010 0.0000         0.0010 0.0020        0.0040 0.0020
pct_dep_esq         0.0070 0.0000         0.0070 0.0020        0.0130 0.0020
comparecimento     -0.0080 0.0000        -0.0080 0.0020       -0.0120 0.0020
delta_comp         -0.0090 0.0000        -0.0090 0.0010       -0.0040 0.0010
log_pop             0.0050 0.0000         0.0050 0.0010        0.0060 0.0020
log_dens            0.0220 0.0000         0.0220 0.0010        0.0330 0.0020
pct_mulheres        0.0300 0.0000         0.0300 0.0020        0.0240 0.0020
pct_16_24          -0.0200 0.0010     

In [6]:
rng = np.random.default_rng(0)
pis = {k: E.CANDIDATOS[k].prever(AJ[k], df, mapa) for k in ["bin_logit", "bb_logit"]}
rq = {"binomial": M.residuos_quantilicos(y, n, pis["bin_logit"], rng=np.random.default_rng(1)),
      "beta-binomial": M.residuos_quantilicos(y, n, pis["bb_logit"], theta=M.theta_de(bb), rng=np.random.default_rng(1))}
G.salvar(G.qq_normal(rq), FIG / "mlg_qq_binomial_vs_bb.html")
G.salvar(G.forest_coeficientes({"binomial": E.tabela_coeficientes(AJ["bin_logit"]), "beta-binomial": E.tabela_coeficientes(AJ["bb_logit"])}),
         FIG / "mlg_forest_binomial_vs_bb.html")
print({k: (round(v.mean(), 3), round(v.std(), 3)) for k, v in rq.items()})

{'binomial': (np.float64(-0.007), np.float64(3.446)), 'beta-binomial': (np.float64(-0.004), np.float64(1.004))}


**Leitura.** Os coeficientes pontuais mudam pouco entre binomial e quase-binomial (são idênticos), mas os erros-padrão do quase-binomial são ≈ √φ vezes maiores; no beta-binomial os pesos
deixam de ser proporcionais a nᵢ (a variância satura em ρ), então municípios grandes deixam de dominar o ajuste e os coeficientes e SEs mudam.
Os resíduos quantílicos (QQ-plot em `results/figures/mlg_qq_binomial_vs_bb.html`) devem ter média ≈ 0 e desvio ≈ 1 se o modelo descreve a dispersão: o binomial puro fica com desvio muito maior que 1.

## Parte 8 - Preditor linear

Variamos o preditor linear sobre o beta-binomial (verossimilhança) e sobre o quase-binomial (mesmo ponto de partida do binomial, SE robustos à dispersão):

- **Offset do lag (modelo de swing):** coeficiente de `logit_pt_2t_2022` fixo em 1, isto é, logit(πᵢ) = logit(share 2022) + xᵢᵀβ. Testa-se contra o coeficiente livre.
- **Spline no lag:** B-spline cúbica com 4 graus de liberdade (`patsy.bs`); valores do teste fora do suporte do treino são limitados à fronteira.
- **Região × lag:** inclinação do lag diferente por região.
- **Efeitos fixos de UF** no lugar da região.
- **GLMM** com intercepto aleatório por UF.
- **Variáveis categóricas** nível a nível, **seleção de variáveis** (passo a passo, Hosmer-Lemeshow e todos os subconjuntos) e o conjunto de registro `bb_selecao` (seções 8.1 a 8.3, ao fim da Parte). A regressão L1 (laço) segue como referência de parcimônia na Parte 9.

### Teoria - Testes de hipóteses sobre coeficientes

**Wald.** $z=\hat\beta_j/\mathrm{EP}(\hat\beta_j)\approx N(0,1)$ sob $H_0:\beta_j=0$; usa só o ajuste do modelo completo, mas é menos confiável que a razão de verossimilhanças quando a verossimilhança é assimétrica.

**Razão de verossimilhanças (LRT).** Para modelos aninhados, $\Delta=2(\ell_1-\ell_0)\sim\chi^2_{\mathrm{gl}}$, com gl = diferença de parâmetros. Para um fator com vários níveis (região, UF) o LRT testa todas as dummies de uma vez; os testes de Wald individuais comparam cada nível só com a referência e não respondem se o fator, como um todo, importa.

**Com sobredispersão.** No quase-binomial, $\Delta D/\hat\phi\sim\chi^2_{\mathrm{gl}}$ aproximadamente e, estimando $\phi$, usa-se o teste F: $F=\dfrac{\Delta D/\mathrm{gl}}{\hat\phi}\sim F_{\mathrm{gl},\,N-p}$, com $\hat\phi$ do modelo maior. O LRT com o binomial puro rejeitaria quase tudo, pois ignora $\phi\approx21{,}6$.

In [7]:
for nome in ["bb_offset", "quasi_offset", "bb_spline", "quasi_spline", "bb_reglag", "quasi_reglag", "bb_uf", "quasi_uf"]:
    ajustar(nome)
# coeficiente livre do lag (escala bruta = coef_z / dp)
dp = AJ["bb_logit"].pad.desvio["lag"]; tb = E.tabela_coeficientes(AJ["bb_logit"]).loc["lag"]
b_raw, ep_raw = tb.coef / dp, tb.ep / dp
print(f"lag livre (beta-binomial): coef bruto = {b_raw:.3f} (EP {ep_raw:.3f}); H0: coef=1 -> z = {(b_raw-1)/ep_raw:.2f}")
tq = E.tabela_coeficientes(AJ["quasi_logit"]).loc["lag"]; bq = tq.coef / dp
print(f"lag livre (quase-binomial): coef bruto = {bq:.3f} (EP {tq.ep/dp:.3f}); z(H0=1) = {(bq-1)/(tq.ep/dp):.2f}")
lr = 2 * (AJ["bb_logit"].loglik - AJ["bb_offset"].loglik)
print(f"LRT offset vs livre (beta-binomial): 2*dlogL = {lr:.1f}, gl=1, p = {stats.chi2.sf(lr, 1):.2e}; dAIC(offset - livre) = {-2*AJ['bb_offset'].loglik+2*AJ['bb_offset'].k - (-2*AJ['bb_logit'].loglik+2*AJ['bb_logit'].k):.1f}")
for a_, b_ in [("bb_logit", "bb_spline"), ("bb_logit", "bb_reglag")]:
    lr2 = 2 * (AJ[b_].loglik - AJ[a_].loglik); gl = AJ[b_].k - AJ[a_].k
    print(f"LRT {a_} vs {b_}: 2*dlogL = {lr2:.1f}, gl={gl}, p = {stats.chi2.sf(lr2, gl):.2e}")
print("quase-binomial (teste F de Wald na deviance escalada):")
for a_, b_ in [("quasi_logit", "quasi_offset"), ("quasi_logit", "quasi_spline"), ("quasi_logit", "quasi_reglag"), ("quasi_logit", "quasi_uf")]:
    da = LINHAS[a_]["deviance"]; db = LINHAS[b_]["deviance"]; ph = AJ["quasi_logit"].extra["phi"]
    gl = abs(AJ[b_].k - AJ[a_].k)
    print(f"  {a_} -> {b_}: |ddeviance| = {abs(da-db):.1f}, gl={gl}, F = {abs(da-db)/gl/ph:.2f}, p = {stats.f.sf(abs(da-db)/gl/ph, gl, N-AJ[b_].k):.2e}")

lag livre (beta-binomial): coef bruto = 0.897 (EP 0.004); H0: coef=1 -> z = -28.89
lag livre (quase-binomial): coef bruto = 0.899 (EP 0.003); z(H0=1) = -29.86
LRT offset vs livre (beta-binomial): 2*dlogL = 774.6, gl=1, p = 1.79e-170; dAIC(offset - livre) = 772.6
LRT bb_logit vs bb_spline: 2*dlogL = 33.6, gl=3, p = 2.39e-07
LRT bb_logit vs bb_reglag: 2*dlogL = 59.4, gl=4, p = 3.91e-12
quase-binomial (teste F de Wald na deviance escalada):
  quasi_logit -> quasi_offset: |ddeviance| = 19331.8, gl=1, F = 887.94, p = 4.08e-181
  quasi_logit -> quasi_spline: |ddeviance| = 474.1, gl=3, F = 7.26, p = 7.41e-05
  quasi_logit -> quasi_reglag: |ddeviance| = 1244.9, gl=4, F = 14.30, p = 1.30e-11
  quasi_logit -> quasi_uf: |ddeviance| = 21828.3, gl=22, F = 45.57, p = 4.75e-181


In [8]:
t8 = pd.DataFrame([LINHAS[k] for k in ["bb_logit", "bb_offset", "bb_spline", "bb_reglag", "bb_uf",
                                         "quasi_logit", "quasi_offset", "quasi_spline", "quasi_reglag", "quasi_uf"]])[cols + ["qaic"]]
print(t8.to_string(index=False))
sp = AJ["bb_spline"]; ts = E.tabela_coeficientes(sp)
print(ts.filter(like="lag_bs", axis=0).round(3).to_string())
tr_ = E.tabela_coeficientes(AJ["bb_reglag"]); print(tr_.filter(like=":lag", axis=0).round(3).to_string())

      modelo ligacao  k       loglik         aic         bic     deviance  deviance_gl  phi_pearson  mae_pond_pp       qaic
    bb_logit   logit 26 -36,327.7143 72,707.4286 72,879.6825 130,791.9089      23.5916      23.6112       1.1548        NaN
   bb_offset   logit 25 -36,715.0175 73,480.0349 73,645.6637 167,052.7443      30.1267      30.3014       1.3498        NaN
   bb_spline   logit 29 -36,310.9069 72,679.8138 72,871.9431 132,126.1755      23.8452      23.8697       1.1657        NaN
   bb_reglag   logit 30 -36,298.0228 72,656.0457 72,854.8002 130,907.7978      23.6296      23.6728       1.1434        NaN
       bb_uf   logit 48 -35,911.7474 71,919.4948 72,237.5020 123,419.7780      22.3506      22.3498       1.1664        NaN
 quasi_logit   logit 26          NaN         NaN         NaN 120,618.7959      21.7566      21.7753       1.0630 7,977.2196
quasi_offset   logit 25          NaN         NaN         NaN 139,950.6104      25.2391      25.3903       1.1548 7,608.3889
quasi_sp

**GLMM.** `BinomialBayesMixedGLM` do statsmodels exige resposta 0/1 e não aceita (y, n) agrupados. Mostramos a falha e usamos um GLMM por PQL próprio
(`ModeloGLMM` em `especificacoes.py`): intercepto aleatório por UF, variância por EM, pesos divididos por φ (equivale a um GLMM quase-binomial).

In [9]:
Xg = E.montar_X(df, mapa, pad, E.Termos(usa_uf=True, uf_aleatoria=True))
try:
    BinomialBayesMixedGLM(y / n, Xg.filter(regex="^(?!uf_)"), Xg.filter(like="uf_").to_numpy(), np.zeros(27, int))
    print("BinomialBayesMixedGLM aceitou a resposta fracionária (inesperado)")
except Exception as e:
    print("BinomialBayesMixedGLM recusou a resposta agrupada/fracionária (exige endog 0/1):", type(e).__name__)
ajustar("glmm_uf")
g = AJ["glmm_uf"]; print({k: round(v, 4) for k, v in g.extra.items()})
ef = pd.Series(g.modelo.beta, index=g.nomes).filter(like="uf_").sort_values()
print("efeitos aleatórios de UF (logit): menores", ef.head(3).round(3).to_dict(), "maiores", ef.tail(3).round(3).to_dict())

BinomialBayesMixedGLM recusou a resposta agrupada/fracionária (exige endog 0/1): ValueError
{'sigma_uf': 0.054, 'phi': 17.8336}
efeitos aleatórios de UF (logit): menores {'uf_GO': -0.092, 'uf_RS': -0.079, 'uf_CE': -0.062} maiores {'uf_RR': 0.084, 'uf_AP': 0.087, 'uf_AC': 0.099}


### 8.1 Variáveis categóricas, nível a nível

Na atividade 1 e 2 cada fator foi lido nível a nível (z contra a referência), testado como um todo (razão de verossimilhanças) e reagrupado quando dois níveis eram indistinguíveis (Paula, *Modelos de regressão com apoio computacional*). Repetimos o roteiro com os dois fatores categóricos do preditor e com a binária de prefeito suplementar, no **beta-binomial completo** (as 18 contínuas). As tabelas descritivas nível a nível que propuseram os testes estão no notebook 01 (Parte 3, subseção "Variáveis categóricas e binárias, nível a nível"). A escala é o logit, então $e^{\beta}$ é a razão de chances contra a referência, com as demais covariáveis fixas.

**Regra (fixada antes de ver os resultados):**

1. Cada fator parte dos seus **níveis brutos**: `bloco_prefeito_2024` (referência direita; esquerda, centro e sem_prefeito) e `regiao` (referência Sudeste).
2. A cada passo os candidatos são todos os pares de níveis fora da referência **e** todo nível fora da referência fundido **na referência**; cada um é avaliado por LRT no beta-binomial completo.
3. Funde-se o candidato de maior p se p > 0,05, reajusta-se e repete-se até que nenhum candidato tenha p > 0,05.
4. A binária `prefeito_suplementar_2024` (os 16 municípios sem prefeito, valor ausente, entram como 0) entra no modelo se o LRT de adicioná-la ao beta-binomial completo, depois das fusões dos fatores, der p ≤ 0,05.

Os erros-padrão de Wald abaixo ignoram a correlação espacial (Parte 10); a significância sob correlação espacial é checada com erros-padrão de Conley no notebook 05.

In [10]:
pad = E.Padronizador(mapa).fit(df)

ajuste_bb = partial(SEL.ajuste_bb, y=y, n=n)
tabela_niveis = SEL.tabela_niveis
fundir = SEL.fundir
def lrt(res_ref, X_ref: pd.DataFrame, X_red: pd.DataFrame) -> tuple[float, int, float]:
    return SEL.lrt(res_ref, X_ref, X_red, y, n)
def analisar_fator(X: pd.DataFrame, cols: list[str], rotulo: str, prefixo: str, ref: str, p_fusao: float = 0.05):
    """Tabela por nível, LRT de retirar o fator e fusão sequencial, entre pares e na referência `ref` (regra: p > p_fusao); lógica em src/selecao.py."""
    return SEL.analisar_fator(X, y, n, cols, rotulo, prefixo, p_fusao, ref=ref)

X_cheio = E.montar_X(df, mapa, pad, E.Termos())
print("Níveis (municípios):", df[mapa["regiao"]].value_counts().to_dict(), df[mapa["bloco_pref"]].value_counts().to_dict())
cols_reg = [c for c in X_cheio.columns if c.startswith("reg_")]
X_reg, res_reg, cols_reg = analisar_fator(X_cheio, cols_reg, "Região (referência Sudeste)", "reg_", "Sudeste")

Níveis (municípios): {'Nordeste': 1794, 'Sudeste': 1668, 'Sul': 1191, 'Centro-Oeste': 467, 'Norte': 450} {'direita': 4770, 'esquerda': 735, 'centro': 49, 'sem_prefeito': 16}


=== Região (referência Sudeste), passo 0: 4 níveis fora da referência; loglik = -36,327.71
                    beta     EP        z      p     OR  OR_inf95  OR_sup95
reg_Centro-Oeste -0.0288 0.0057  -5.0322 0.0000 0.9716    0.9608    0.9826
reg_Nordeste      0.0132 0.0060   2.2130 0.0269 1.0133    1.0015    1.0252
reg_Norte         0.0139 0.0075   1.8461 0.0649 1.0140    0.9991    1.0291
reg_Sul          -0.0680 0.0048 -14.1990 0.0000 0.9342    0.9255    0.9431


LRT (todos os níveis = 0, gl=4): Δ = 233.4, p = 2.41e-49


LRT de igualdade entre níveis (H0: β_a = β_b):
     nivel_a  nivel_b    delta  gl      p
Centro-Oeste Nordeste  32.5289   1 0.0000
Centro-Oeste    Norte  34.9160   1 0.0000
Centro-Oeste      Sul  34.8281   1 0.0000
    Nordeste    Norte   0.0098   1 0.9210
    Nordeste      Sul 130.0327   1 0.0000
       Norte      Sul  93.8119   1 0.0000
Centro-Oeste  Sudeste  25.2856   1 0.0000
    Nordeste  Sudeste   4.8934   1 0.0270
       Norte  Sudeste   3.4055   1 0.0650
         Sul  Sudeste 197.9374   1 0.0000
Funde Nordeste e Norte (p = 0.921 > 0.05) e reajusta.


=== Região (referência Sudeste), passo 1: 3 níveis fora da referência; loglik = -36,327.72
                      beta     EP        z      p     OR  OR_inf95  OR_sup95
reg_Centro-Oeste   -0.0290 0.0055  -5.2928 0.0000 0.9715    0.9611    0.9819
reg_Sul            -0.0680 0.0048 -14.1956 0.0000 0.9342    0.9255    0.9431
reg_Nordeste+Norte  0.0134 0.0056   2.3742 0.0176 1.0135    1.0023    1.0248


LRT (todos os níveis = 0, gl=3): Δ = 233.4, p = 2.52e-50


LRT de igualdade entre níveis (H0: β_a = β_b):
       nivel_a        nivel_b    delta  gl      p
  Centro-Oeste            Sul  36.7946   1 0.0000
  Centro-Oeste Nordeste+Norte  43.5529   1 0.0000
           Sul Nordeste+Norte 141.4021   1 0.0000
  Centro-Oeste        Sudeste  27.9582   1 0.0000
           Sul        Sudeste 197.9562   1 0.0000
Nordeste+Norte        Sudeste   5.6326   1 0.0176
Nenhum candidato com p > 0.05: nenhum nível é fundido.


**Leitura da região** (beta-binomial completo, referência Sudeste, demais covariáveis fixas).

- O fator importa: retirar a região inteira custa Δ = 233,4 de deviance (gl = 4, p = 2,4e-49).
- Centro-Oeste: razão de chances 0,972 (IC95% 0,961 a 0,983); a chance de voto em Lula é cerca de 2,8% menor que no Sudeste, com o mesmo perfil e o mesmo resultado de 2022.
- Sul: 0,934 (0,926 a 0,943), chance cerca de 6,6% menor que no Sudeste.
- Nordeste: 1,013 (1,002 a 1,025) e Norte: 1,014 (0,999 a 1,029). Ambos ficam perto do Sudeste, porque o `lag` já carrega o nível regional de 2022; a região só corrige o resíduo.
- Passo 0, candidatos entre pares: **Nordeste = Norte não é rejeitada** (Δ = 0,010; p = 0,921); todos os outros pares são rejeitados (p < 1e-8; o maior entre eles é Centro-Oeste × Nordeste, Δ = 32,5).
- Passo 0, candidatos na referência: Centro-Oeste = Sudeste (Δ = 25,3) e Sul = Sudeste (Δ = 197,9) são rejeitados (p < 1e-6); Nordeste = Sudeste dá Δ = 4,89 (p = 0,027) e Norte = Sudeste dá Δ = 3,41 (p = 0,065). O maior p do passo é o de Nordeste × Norte, então a regra funde esse par.
- Após a fusão: Centro-Oeste 0,972, Sul 0,934 e Nordeste+Norte **1,0135** (1,002 a 1,025; p = 0,018). A log-verossimilhança praticamente não muda (-36.327,71 para -36.327,72), então a fusão economiza um parâmetro sem perda de ajuste.
- Passo 1, a regra para: entre pares, Δ = 36,8 (Centro-Oeste × Sul), 43,6 e 141,4 (todos p < 1e-8); na referência, Centro-Oeste = Sudeste dá Δ = 28,0, Sul = Sudeste dá Δ = 198,0 e Nordeste+Norte = Sudeste dá Δ = 5,63 (p = 0,018). Nenhum candidato tem p > 0,05, então nenhum nível é fundido nem entra na referência: o nível Nordeste+Norte, por pouco, se distingue do Sudeste.
- Tamanhos: Nordeste 1.794 municípios, Sudeste 1.668, Sul 1.191, Centro-Oeste 467, Norte 450.

A fusão `Nordeste + Norte` vale para todo o restante do projeto (`FUNDIR`, `Termos.regiao_fundir`).

In [11]:
# bloco parte dos 4 níveis brutos (direita = referência); a regra decide fusões entre níveis e na referência
BLOCO_BRUTO = (("esquerda",), ("centro",), ("sem_prefeito",))
FUNDIR = tuple(tuple(c[4:].split("+")) for c in cols_reg if "+" in c)
X_bru = E.montar_X(df, mapa, pad, replace(E.Termos(regiao_fundir=FUNDIR), bloco_grupos=BLOCO_BRUTO))
cols_bru = [c for c in X_bru.columns if c.startswith("bloco_")]
X_blo, res_blo, cols_blo = analisar_fator(X_bru, cols_bru, "Bloco do prefeito (referência direita)", "bloco_", "direita")
BLOCO = tuple(tuple(c[6:].split("+")) for c in cols_blo)
BLOCO = tuple(tuple(x for g in grupo for x in (("centro", "sem_prefeito") if g == "outro" else (g,))) for grupo in BLOCO)
print("\nGrupos do bloco adotados:", BLOCO, "(níveis fora dos grupos = referência direita)")

# binária: prefeito eleito em eleição suplementar (None = 16 municípios sem prefeito, codificados 0)
T_base = E.Termos(regiao_fundir=FUNDIR, bloco_grupos=BLOCO)
X_base = E.montar_X(df, mapa, pad, T_base)
X_sup = E.montar_X(df, mapa, pad, replace(T_base, usa_suplementar=True))
r_sup = ajuste_bb(X_sup)
print("\n=== Prefeito suplementar 2024 (binária), adicionada ao beta-binomial após as fusões; loglik =", f"{r_sup.llf:,.2f}")
print(tabela_niveis(r_sup, X_sup, ["prefeito_suplementar"]).round(4).to_string())
d_s, gl_s, p_s = lrt(r_sup, X_sup, X_base)
SUPL = bool(p_s <= 0.05)
print(f"LRT de adicionar a suplementar: Δ = {d_s:.2f}, gl={gl_s}, p = {p_s:.3f} -> {'entra' if SUPL else 'não entra'}")
print("Municípios suplementar:", int(X_sup["prefeito_suplementar"].sum()))

TERMOS_SEL = E.Termos(regiao_fundir=FUNDIR, bloco_grupos=BLOCO, usa_suplementar=SUPL)
Xc = E.montar_X(df, mapa, pad, TERMOS_SEL)
assert set(cols_reg) <= set(Xc.columns) and np.allclose(Xc[cols_reg].to_numpy(), X_reg[cols_reg].to_numpy())
print("\nFusões adotadas na região:", FUNDIR or "nenhuma", "| colunas de região no modelo:", cols_reg)
print("Bloco:", BLOCO, "| colunas de bloco:", cols_blo, "| suplementar:", SUPL)
print("TERMOS_SEL igual ao padrão com FUNDIR:", TERMOS_SEL == E.Termos(regiao_fundir=FUNDIR))

=== Bloco do prefeito (referência direita), passo 0: 3 níveis fora da referência; loglik = -36,327.45
                      beta     EP       z      p     OR  OR_inf95  OR_sup95
bloco_esquerda     -0.0165 0.0041 -4.0834 0.0000 0.9836    0.9758    0.9914
bloco_centro        0.0175 0.0125  1.3925 0.1638 1.0176    0.9929    1.0430
bloco_sem_prefeito  0.0362 0.0224  1.6150 0.1063 1.0368    0.9923    1.0834


LRT (todos os níveis = 0, gl=3): Δ = 22.0, p = 6.41e-05


LRT de igualdade entre níveis (H0: β_a = β_b):
     nivel_a      nivel_b   delta  gl      p
    esquerda       centro  6.8186   1 0.0090
    esquerda sem_prefeito  5.4335   1 0.0198
      centro sem_prefeito  0.5321   1 0.4657
    esquerda      direita 16.6333   1 0.0000
      centro      direita  1.9401   1 0.1637
sem_prefeito      direita  2.6104   1 0.1062
Funde centro e sem_prefeito (p = 0.466 > 0.05) e reajusta.


=== Bloco do prefeito (referência direita), passo 1: 2 níveis fora da referência; loglik = -36,327.72
                             beta     EP       z      p     OR  OR_inf95  OR_sup95
bloco_esquerda            -0.0166 0.0041 -4.0879 0.0000 0.9836    0.9758    0.9914
bloco_centro+sem_prefeito  0.0219 0.0110  1.9960 0.0459 1.0221    1.0004    1.0444


LRT (todos os níveis = 0, gl=2): Δ = 21.5, p = 2.14e-05


LRT de igualdade entre níveis (H0: β_a = β_b):
            nivel_a             nivel_b   delta  gl      p
           esquerda centro+sem_prefeito 11.1904   1 0.0008
           esquerda             direita 16.6713   1 0.0000
centro+sem_prefeito             direita  3.9937   1 0.0457
Nenhum candidato com p > 0.05: nenhum nível é fundido.

Grupos do bloco adotados: (('esquerda',), ('centro', 'sem_prefeito')) (níveis fora dos grupos = referência direita)



=== Prefeito suplementar 2024 (binária), adicionada ao beta-binomial após as fusões; loglik = -36,327.62
                        beta     EP       z      p     OR  OR_inf95  OR_sup95
prefeito_suplementar -0.0077 0.0169 -0.4561 0.6483 0.9923    0.9599    1.0258


LRT de adicionar a suplementar: Δ = 0.21, gl=1, p = 0.649 -> não entra
Municípios suplementar: 27

Fusões adotadas na região: (('Nordeste', 'Norte'),) | colunas de região no modelo: ['reg_Centro-Oeste', 'reg_Sul', 'reg_Nordeste+Norte']
Bloco: (('esquerda',), ('centro', 'sem_prefeito')) | colunas de bloco: ['bloco_esquerda', 'bloco_centro+sem_prefeito'] | suplementar: False
TERMOS_SEL igual ao padrão com FUNDIR: True


**Leitura do bloco do prefeito** (referência direita, demais covariáveis fixas; a regra parte dos 3 níveis brutos fora da referência).

- Níveis brutos (n): direita 4.770 (referência), esquerda 735, centro 49, sem prefeito 16. O fator importa: retirar o bloco inteiro custa Δ = 22,0 (gl = 3, p = 6,4e-5).
- Esquerda: razão de chances 0,984 (IC95% 0,976 a 0,991); a chance de voto em Lula é cerca de 1,6% menor que sob prefeito de direita, com o mesmo perfil e o mesmo `lag`. O efeito do bloco já passa em boa parte pelo `lag` e por `pct_dep_esq`; o que sobra é pequeno.
- Centro: 1,018 (0,993 a 1,043; p = 0,164). Sem prefeito: 1,037 (0,992 a 1,083; p = 0,106). Individualmente, nenhum dos dois se distingue da direita.
- Passo 0, candidatos: **centro = sem prefeito não é rejeitada** (Δ = 0,53; p = 0,466), o maior p do passo; esquerda × centro (Δ = 6,8; p = 0,009) e esquerda × sem prefeito (Δ = 5,4; p = 0,020) são rejeitadas. Na referência: centro = direita dá Δ = 1,94 (p = 0,164) e sem prefeito = direita dá Δ = 2,61 (p = 0,106), ambos abaixo do p de centro × sem prefeito; esquerda = direita é rejeitada (Δ = 16,6). A regra funde centro e sem prefeito. São só 49 municípios de centro e 16 sem prefeito, então o teste tem pouco poder para distingui-los.
- Passo 1: o grupo centro+sem_prefeito tem razão de chances 1,022 (1,000 a 1,044; p = 0,046). Candidatos: esquerda × centro+sem_prefeito, Δ = 11,2 (p = 8e-4); esquerda = direita, Δ = 16,7 (p < 1e-4); centro+sem_prefeito = direita, Δ = 3,99 (p = 0,046). Nenhum tem p > 0,05, então a regra para: o grupo não é fundido na direita, por pouco. A log-verossimilhança fica em -36.327,72 (contra -36.327,45 com os 3 níveis separados), sem perda de ajuste com um parâmetro a menos.
- Binária `prefeito_suplementar_2024` (27 municípios com prefeito de eleição suplementar; os 16 sem prefeito entram como 0), adicionada depois das fusões: razão de chances 0,992 (0,960 a 1,026); LRT de adicioná-la Δ = 0,21 (gl = 1, p = 0,649). Como p > 0,05, ela **não entra** no modelo.

Conclusão: o bloco mantém dois grupos, esquerda e centro+sem prefeito (a codificação `outro` da Parte 6), agora obtidos pela regra a partir dos níveis brutos; a suplementar fica de fora e a região usa a fusão Nordeste + Norte. `TERMOS_SEL` coincide com `Termos(regiao_fundir=FUNDIR)`, então os resultados das células seguintes não mudam. A fusão Nordeste + Norte vale para o restante do projeto.

### 8.2 Seleção das contínuas: passo a passo e todos os subconjuntos

Com os fatores decididos em 8.1 (região e bloco ficam sempre no modelo, e o `lag` também), escolhemos entre as 17 contínuas restantes por seis buscas gulosas no beta-binomial (backward, forward e both, cada uma por BIC e por AIC), pelo **stepwise de Hosmer-Lemeshow** (Paula, seção 4.6.9: entra a variável de menor π se π < 0,20; sai a de maior π se π > 0,25; π = P(χ²ₘ > Δ), com Δ a diferença de deviance) e por **todos os subconjuntos** ($2^{17}=131.072$ ajustes: triagem pelo QBIC quase-binomial e refit beta-binomial dos 200 melhores). A busca exaustiva é a referência: diz quanto cada método perde. Em todas as comparações o conjunto exclui o `lag` (sempre presente).

In [12]:
CAND = SEL.CANDIDATAS_PADRAO
RES, TRILHAS = {}, {}
t_sel = time.time()
for crit in ("bic", "aic"):
    for met in ("backward", "forward", "both"):
        t0 = time.time()
        vs_, tr_ = SEL.passo_a_passo(df, mapa, TERMOS_SEL, met, crit)
        RES[f"{met}_{crit}"], TRILHAS[f"{met}_{crit}"] = (vs_,), tr_
        print(f"=== {met} por {crit.upper()} ({time.time()-t0:.0f}s): {len(vs_)} contínuas; fora: {[v for v in CAND if v not in vs_]}")
        print(tr_.round(2).to_string(index=False))


=== backward por BIC (95s): 11 contínuas; fora: ['nota_dep', 'delta_comp', 'log_pop', 'pct_60_mais', 'pib_pc', 'pct_urbana']
 passo   acao    variavel  valor_criterio  n_vars
     0 inicio                 72,871.0700      17
     1    sai pct_60_mais     72,862.4500      16
     2    sai      pib_pc     72,854.9700      15
     3    sai    nota_dep     72,850.0700      14
     4    sai  pct_urbana     72,848.7800      13
     5    sai     log_pop     72,846.3400      12
     6    sai  delta_comp     72,844.8700      11


=== forward por BIC (62s): 11 contínuas; fora: ['nota_dep', 'delta_comp', 'log_pop', 'pct_60_mais', 'pib_pc', 'pct_urbana']
 passo   acao          variavel  valor_criterio  n_vars
     0 inicio                       75,397.1800       0
     1  entra      pct_mulheres     73,877.6800       1
     2  entra     cobertura_pbf     73,431.1000       2
     3  entra          log_dens     73,097.2300       3
     4  entra    comparecimento     73,015.9000       4
     5  entra       pct_dep_esq     72,946.6400       5
     6  entra      pct_vab_agro     72,912.4100       6
     7  entra pct_pretos_pardos     72,882.6700       7
     8  entra    pct_analfabeto     72,860.5900       8
     9  entra         pct_16_24     72,857.4900       9
    10  entra   pct_evangelicos     72,849.4700      10
    11  entra      pct_superior     72,844.8700      11


=== both por BIC (78s): 11 contínuas; fora: ['nota_dep', 'delta_comp', 'log_pop', 'pct_60_mais', 'pib_pc', 'pct_urbana']
 passo   acao          variavel  valor_criterio  n_vars
     0 inicio                       75,397.1800       0
     1  entra      pct_mulheres     73,877.6800       1
     2  entra     cobertura_pbf     73,431.1000       2
     3  entra          log_dens     73,097.2300       3
     4  entra    comparecimento     73,015.9000       4
     5  entra       pct_dep_esq     72,946.6400       5
     6  entra      pct_vab_agro     72,912.4100       6
     7  entra pct_pretos_pardos     72,882.6700       7
     8  entra    pct_analfabeto     72,860.5900       8
     9  entra         pct_16_24     72,857.4900       9
    10  entra   pct_evangelicos     72,849.4700      10
    11  entra      pct_superior     72,844.8700      11


=== backward por AIC (48s): 15 contínuas; fora: ['pct_60_mais', 'pib_pc']
 passo   acao    variavel  valor_criterio  n_vars
     0 inicio                 72,705.4400      17
     1    sai pct_60_mais     72,703.4400      16
     2    sai      pib_pc     72,702.5900      15


=== forward por AIC (44s): 15 contínuas; fora: ['pct_60_mais', 'pib_pc']
 passo   acao          variavel  valor_criterio  n_vars
     0 inicio                       75,344.1800       0
     1  entra      pct_mulheres     73,818.0500       1
     2  entra     cobertura_pbf     73,364.8500       2
     3  entra          log_dens     73,024.3500       3
     4  entra    comparecimento     72,936.3900       4
     5  entra       pct_dep_esq     72,860.5100       5
     6  entra      pct_vab_agro     72,819.6600       6
     7  entra pct_pretos_pardos     72,783.3000       7
     8  entra    pct_analfabeto     72,754.5800       8
     9  entra         pct_16_24     72,744.8600       9
    10  entra   pct_evangelicos     72,730.2100      10
    11  entra      pct_superior     72,719.0000      11
    12  entra        delta_comp     72,713.8400      12
    13  entra           log_pop     72,709.6500      13
    14  entra        pct_urbana     72,704.3200      14
    15  entra          nota_dep

=== both por AIC (83s): 15 contínuas; fora: ['pct_60_mais', 'pib_pc']
 passo   acao          variavel  valor_criterio  n_vars
     0 inicio                       75,344.1800       0
     1  entra      pct_mulheres     73,818.0500       1
     2  entra     cobertura_pbf     73,364.8500       2
     3  entra          log_dens     73,024.3500       3
     4  entra    comparecimento     72,936.3900       4
     5  entra       pct_dep_esq     72,860.5100       5
     6  entra      pct_vab_agro     72,819.6600       6
     7  entra pct_pretos_pardos     72,783.3000       7
     8  entra    pct_analfabeto     72,754.5800       8
     9  entra         pct_16_24     72,744.8600       9
    10  entra   pct_evangelicos     72,730.2100      10
    11  entra      pct_superior     72,719.0000      11
    12  entra        delta_comp     72,713.8400      12
    13  entra           log_pop     72,709.6500      13
    14  entra        pct_urbana     72,704.3200      14
    15  entra          nota_dep   

**Leitura da seleção passo a passo** (beta-binomial, 17 contínuas candidatas mais o `lag`, que fica sempre).

- Por **BIC**, retroceder, avançar e o misto chegam ao **mesmo conjunto de 11 contínuas**, com BIC = 72.844,87 (o modelo cheio tem 72.871,07, ou seja 26,2 pontos acima). Saem `nota_dep`, `delta_comp`, `log_pop`, `pct_60_mais`, `pib_pc` e `pct_urbana`. O avanço entra por `pct_mulheres`, `cobertura_pbf`, `log_dens`, `comparecimento`, `pct_dep_esq` e `pct_vab_agro` (os 6 primeiros, que sozinhos baixam o BIC de 75.397 para 72.912) e a última, `pct_superior`, melhora só 4,6 pontos.
- Por **AIC**, os três métodos ficam com **15 contínuas** e AIC = 72.702,59 (cheio: 72.705,44): saem apenas `pct_60_mais` e `pib_pc` (os ganhos de retirar são 2,0 e 0,9 pontos). O AIC penaliza menos e aceita `delta_comp`, `log_pop`, `pct_urbana` e `nota_dep`, cada uma com ganho de 2 a 5 pontos.
- Os métodos concordam entre si dentro de cada critério; a diferença está no critério, não no método. Com N = 5.570 e penalização de log N por parâmetro, o BIC corta variáveis cujo ganho é pequeno.

In [13]:
t0 = time.time()
vs_hl, tab_hl = SEL.stepwise_hosmer_lemeshow(df, mapa, TERMOS_SEL, p_entrada=0.20, p_saida=0.25)
print(f"Hosmer-Lemeshow, etapa 1: efeitos principais ({time.time()-t0:.0f}s). π = P(χ²_gl > Δ); Δ = 2(l_com − l_sem) (diferença de deviance); entra se π < 0,20, sai se π > 0,25.")
print(tab_hl[["iteracao", "acao", "rotulo", "delta_deviance", "gl", "pi"]].round(4).to_string(index=False))
# etapa 2: interações entre pares. Por custo (cada iteração reajusta todos os pares), só entre o lag e os 6 efeitos que entraram primeiro
t0 = time.time()
base2 = tuple(tab_hl.variavel[:6])
_, tab_hl2 = SEL.stepwise_hosmer_lemeshow(df, mapa, TERMOS_SEL, candidatas=base2, p_entrada=0.20, p_saida=0.25, interacoes=True)
t2 = tab_hl2[tab_hl2.etapa == 2]
print(f"\nHosmer-Lemeshow, etapa 2: interações entre lag e {list(base2)} ({time.time()-t0:.0f}s)")
print(t2[["iteracao", "acao", "rotulo", "delta_deviance", "gl", "pi"]].round(4).to_string(index=False) if len(t2) else "nenhuma interação entra (todas com π >= 0,20)")
RES["hl"] = (tuple(vs_hl),)
print(f"\nEfeitos principais finais: {len(vs_hl)}; interações que entraram: {list(t2.rotulo[t2.acao == 'entra']) or 'nenhuma'}")


Hosmer-Lemeshow, etapa 1: efeitos principais (155s). π = P(χ²_gl > Δ); Δ = 2(l_com − l_sem) (diferença de deviance); entra se π < 0,20, sai se π > 0,25.
 iteracao  acao            rotulo  delta_deviance  gl     pi
        1 entra          log_dens      1,426.5830   1 0.0000
        2 entra     cobertura_pbf        417.8472   1 0.0000
        3 entra      pct_mulheres        481.3980   1 0.0000
        4 entra    comparecimento         89.9599   1 0.0000
        5 entra       pct_dep_esq         77.8842   1 0.0000
        6 entra      pct_vab_agro         42.8480   1 0.0000
        7 entra pct_pretos_pardos         38.3671   1 0.0000
        8 entra    pct_analfabeto         30.7120   1 0.0000
        9 entra         pct_16_24         11.7237   1 0.0006
       10 entra   pct_evangelicos         16.6447   1 0.0000
       11 entra      pct_superior         13.2189   1 0.0003
       12 entra        delta_comp          7.1584   1 0.0075
       13 entra           log_pop          6.1839   1 


Hosmer-Lemeshow, etapa 2: interações entre lag e ['log_dens', 'cobertura_pbf', 'pct_mulheres', 'comparecimento', 'pct_dep_esq', 'pct_vab_agro'] (618s)
 iteracao  acao                       rotulo  delta_deviance  gl     pi
        1 entra cobertura_pbf:comparecimento        122.3691   1 0.0000
        2 entra              lag:pct_dep_esq         46.7121   1 0.0000
        3 entra  comparecimento:pct_vab_agro         39.6598   1 0.0000
        4 entra            lag:cobertura_pbf         14.8404   1 0.0001
        5 entra   cobertura_pbf:pct_mulheres         11.4213   1 0.0007
        6 entra  pct_mulheres:comparecimento          8.0532   1 0.0045
        7 entra           lag:comparecimento          8.0992   1 0.0044
        8 entra         log_dens:pct_dep_esq          7.2042   1 0.0073
        9 entra        log_dens:pct_mulheres          8.2174   1 0.0041
       10 entra    pct_mulheres:pct_vab_agro         10.7280   1 0.0011
       11 entra     pct_dep_esq:pct_vab_agro          4.

**Leitura do Hosmer-Lemeshow (Paula, 4.6.9)** (entra se π < 0,20; sai se π > 0,25, com π = P(χ² > Δ) e Δ a diferença de deviance).

- Etapa 1, efeitos principais: entram 15 variáveis, as mesmas do AIC. Os maiores saltos são `log_dens` (Δ = 1.426,6), `pct_mulheres` (481,4) e `cobertura_pbf` (417,8); a última, `nota_dep`, entra com π = 0,054. Com o limiar de 0,20 o critério é mais frouxo que o BIC.
- Etapa 2, interações: **desvio do procedimento por custo.** Em vez de todos os pares de 15 variáveis, testamos as interações apenas entre o `lag` e as 6 primeiras variáveis que entraram (`log_dens`, `cobertura_pbf`, `pct_mulheres`, `comparecimento`, `pct_dep_esq`, `pct_vab_agro`); a etapa com todos os pares passou de 20 minutos sem terminar, e esta leva 448 s.
- Entram **14 interações** em 14 iterações. A maior é `cobertura_pbf:comparecimento` (Δ = 122,4), seguida de `lag:pct_dep_esq` (46,7) e `comparecimento:pct_vab_agro` (39,7). As últimas já são fracas (π entre 0,04 e 0,15), efeito do limiar de 0,20.
- A linha `hl` da comparação (8.3) usa só os 15 efeitos principais, igual ao conjunto do AIC, porque as interações não entram no conjunto de variáveis da especificação final; elas ficam como evidência de que a resposta a algumas variáveis muda com o resultado de 2022 e com a mobilização, a ser tratada, se for o caso, no notebook 05.

In [14]:
TODOS = SEL.todos_os_subconjuntos(df, mapa, TERMOS_SEL, top=200)
print(f"triagem quase-binomial de 2^{len(CAND)} = {2**len(CAND):,} subconjuntos: {TODOS.attrs['tempo_triagem_s']:.0f}s; refit beta-binomial dos 200 melhores por QBIC")
mt = TODOS.sort_values("bic_bb").iloc[0]; ma = TODOS.sort_values("aic_bb").iloc[0]
RES["todos"], RES["todos_aic"] = (tuple(mt.variaveis),), (tuple(ma.variaveis),)
print(TODOS.sort_values("bic_bb").head(8)[["n_vars", "qbic", "bic_bb", "aic_bb", "rank_qbic"]].round(1).to_string())
print(f"melhor BIC_bb: {mt.bic_bb:,.1f} ({int(mt.n_vars)} contínuas, posto QBIC {int(mt.rank_qbic)}); melhor AIC_bb: {ma.aic_bb:,.1f} ({int(ma.n_vars)} contínuas, posto QBIC {int(ma.rank_qbic)})")
print("tempo total da seleção:", round(time.time() - t_sel), "s")


triagem de 131072 subconjuntos: 178.9 s (phi cheio = 21.861)


triagem quase-binomial de 2^17 = 131,072 subconjuntos: 179s; refit beta-binomial dos 200 melhores por QBIC
   n_vars       qbic      bic_bb      aic_bb  rank_qbic
0      13 8,125.3000 72,848.8000 72,709.7000         36
1      13 8,140.8000 72,848.8000 72,709.7000        141
2      13 8,110.0000 72,848.9000 72,709.7000          3
3      13 8,115.0000 72,849.3000 72,710.1000         11
4      14 8,110.6000 72,850.1000 72,704.3000          4
5      12 8,125.4000 72,850.5000 72,718.0000         38
6      12 8,116.7000 72,851.2000 72,718.7000         12
7      12 8,140.7000 72,851.4000 72,718.9000        140
melhor BIC_bb: 72,848.8 (13 contínuas, posto QBIC 36); melhor AIC_bb: 72,702.6 (15 contínuas, posto QBIC 17)
tempo total da seleção: 1493 s


**Leitura de todos os subconjuntos.**

- A triagem quase-binomial de 2^17 = 131.072 subconjuntos leva 183 s (φ do modelo cheio = 21,86). Como o quase-binomial não tem verossimilhança, a triagem usa um QBIC e refaz os **200 melhores** em beta-binomial. O tempo total da seleção (todos os métodos) foi 1.084 s.
- Melhor BIC beta-binomial: **72.848,8, com 13 contínuas** (as 11 do BIC passo a passo mais `delta_comp` e `log_pop`), 3,9 pontos **acima** do passo a passo (72.844,87). O conjunto do passo a passo não entrou entre os 200 refeitos (a triagem por QBIC é só uma aproximação do BIC beta-binomial, e o melhor delas ocupa o posto 36 do QBIC); portanto o passo a passo achou um conjunto melhor que o da busca exaustiva aproximada.
- Melhor AIC beta-binomial: **72.702,6 com 15 contínuas** (posto 17 no QBIC), o mesmo conjunto do AIC passo a passo.
- Conclusão: a triagem de todos os subconjuntos (quase-binomial, 200 refeitos em beta-binomial; não é exaustiva no beta-binomial) não encontra nada melhor que o passo a passo por BIC dentro da amostra e confirma o conjunto do AIC.

### 8.3 Comparação dos métodos e conjunto de registro

Para cada método: BIC e AIC beta-binomiais (dentro da amostra) e o erro médio por UF na validação *leave-one-UF-out* (cada UF é prevista por um ajuste sem ela; o peso das UFs retidas é o n̂ estimado, sem usar o 2º turno de 2026; mesma métrica do notebook 03). **Regra de registro:** vale o conjunto de menor BIC beta-binomial entre os métodos; ele entra em `CANDIDATOS` como `bb_selecao`.

In [15]:
ANTIGO_12 = ("pct_dep_esq", "comparecimento", "log_dens", "pct_mulheres", "pct_16_24", "pct_superior",
             "pct_analfabeto", "cobertura_pbf", "pct_vab_agro", "pct_evangelicos", "pct_pretos_pardos")  # backward antigo (11 sem o lag)
RES["cheio"], RES["antigo_12"] = (tuple(CAND),), (ANTIGO_12,)
CMP = SEL.comparar_metodos(RES, df, mapa, TERMOS_SEL, referencia="todos")
uf = df[mapa["uf"]].astype(str).to_numpy(); folds = M.folds_por_grupo(uf)
obs_uf = pd.Series(y).groupby(uf).sum() / pd.Series(n).groupby(uf).sum()
distintos: dict[frozenset, str] = {}
for k in CMP.metodo: distintos.setdefault(frozenset(RES[k][0]), f"sel_{len(distintos)}")
for conj, nome in distintos.items():
    E.EXTENSOES[nome] = E.Candidato(nome, "betabin", "logit", E.Termos(vars=tuple(v for v in E.CONTINUAS if v in conj or v == "lag"), regiao_fundir=FUNDIR))
E.definir_dados_cv(df, mapa); n_hat = E.n_loso(df)
t0 = time.time()
with ProcessPoolExecutor(max_workers=14, mp_context=get_context("fork")) as ex:
    fut = {(nome, i): ex.submit(E.prever_fold, nome, tr, te) for nome in distintos.values() for i, (tr, te) in enumerate(folds)}
    PI_LOSO = {}
    for nome in distintos.values():
        pi = np.full(len(df), np.nan)
        for i, (tr, te) in enumerate(folds): pi[te] = fut[(nome, i)].result()
        PI_LOSO[nome] = pi
for nome in distintos.values(): E.EXTENSOES.pop(nome)
mae = {nome: float(((M.agregar(pi, n_hat, uf).drop("Brasil") - obs_uf).abs() * 100).mean()) for nome, pi in PI_LOSO.items()}
CMP["mae_uf_loso_pp"] = [mae[distintos[frozenset(RES[k][0])]] for k in CMP.metodo]
print(f"LOSO de {len(distintos)} conjuntos distintos ({time.time()-t0:.0f}s)")
CMP["dBIC_vs_todos"] = CMP.BIC_bb - CMP.BIC_bb[CMP.metodo == "todos"].iloc[0]
vis = CMP.assign(variaveis=CMP.variaveis.map(len)).rename(columns={"variaveis": "n_cont"})
print(vis[["metodo", "n_vars", "BIC_bb", "AIC_bb", "dBIC_vs_todos", "mae_uf_loso_pp"]].round(3).to_string(index=False))
for _, r in CMP.iterrows(): print(f"{r.metodo:>13}: a mais {list(r.a_mais_vs_ref)} | a menos {list(r.a_menos_vs_ref)}")
CMP.assign(variaveis=CMP.variaveis.map(", ".join), a_mais_vs_ref=CMP.a_mais_vs_ref.map(", ".join), a_menos_vs_ref=CMP.a_menos_vs_ref.map(", ".join)).to_csv(TAB / "selecao_metodos.csv", index=False)


LOSO de 4 conjuntos distintos (71s)
      metodo  n_vars      BIC_bb      AIC_bb  dBIC_vs_todos  mae_uf_loso_pp
backward_bic      11 72,844.8740 72,718.9960        -3.9080          1.0350
 forward_bic      11 72,844.8740 72,718.9960        -3.9080          1.0350
    both_bic      11 72,844.8740 72,718.9960        -3.9080          1.0350
backward_aic      15 72,854.9680 72,702.5900         6.1870          1.0030
 forward_aic      15 72,854.9680 72,702.5900         6.1870          1.0030
    both_aic      15 72,854.9680 72,702.5900         6.1870          1.0030
          hl      15 72,854.9680 72,702.5900         6.1870          1.0030
       todos      13 72,848.7820 72,709.6540         0.0000          1.0000
   todos_aic      15 72,854.9680 72,702.5900         6.1870          1.0030
       cheio      17 72,871.0670 72,705.4380        22.2860          1.0120
   antigo_12      11 72,844.8740 72,718.9960        -3.9080          1.0350
 backward_bic: a mais [] | a menos ['delta_comp', 'l

In [16]:
# regra de registro: menor BIC_bb entre os métodos de seleção (as linhas "cheio" e "antigo_12" são só referência)
metodos_sel = CMP[~CMP.metodo.isin(["cheio", "antigo_12"])]
venc = metodos_sel.sort_values("BIC_bb").iloc[0]
vs_rec = list(venc.variaveis)
E.registrar_selecao(vs_rec, FUNDIR); ajustar("bb_selecao")
print(f"conjunto de registro: {venc.metodo} | {len(vs_rec)} contínuas além do lag | BIC_bb = {venc.BIC_bb:,.1f}")
print("fora:", [v for v in CAND if v not in vs_rec], "| a mais que o antigo:", sorted(set(vs_rec) - set(ANTIGO_12)), "| a menos:", sorted(set(ANTIGO_12) - set(vs_rec)))
json.dump({"metodo": venc.metodo, "variaveis": ["lag"] + vs_rec, "regiao_fundir": [list(g) for g in FUNDIR],
           "BIC_bb": float(venc.BIC_bb), "AIC_bb": float(venc.AIC_bb), "mae_uf_loso_pp": float(venc.mae_uf_loso_pp)},
          open(TAB / "selecao_variaveis.json", "w"), indent=1, ensure_ascii=False)


conjunto de registro: backward_bic | 11 contínuas além do lag | BIC_bb = 72,844.9
fora: ['nota_dep', 'delta_comp', 'log_pop', 'pct_60_mais', 'pib_pc', 'pct_urbana'] | a mais que o antigo: [] | a menos: []


**Leitura da comparação e do conjunto de registro.**

- **Venceu o menor BIC beta-binomial: `backward_bic` (igual ao avanço e ao misto), 11 contínuas além do `lag`, BIC = 72.844,9.** Esse é o conjunto registrado em `bb_selecao`, com a fusão Nordeste + Norte (`selecao_variaveis.json`). Variáveis: `pct_dep_esq`, `comparecimento`, `log_dens`, `pct_mulheres`, `pct_16_24`, `pct_superior`, `pct_analfabeto`, `cobertura_pbf`, `pct_vab_agro`, `pct_evangelicos` e `pct_pretos_pardos`.
- **É o mesmo conjunto do antigo de 12 variáveis** (11 contínuas e o `lag`): nada foi acrescentado nem retirado. O que muda é a especificação em volta (fusão Nordeste + Norte e a regra de registro definida antes dos resultados).
- Conjuntos distintos, dentro da amostra e fora dela (erro médio por UF no leave-one-UF-out):

| conjunto | n contínuas | BIC | AIC | MAE por UF (p.p.) |
|---|---|---|---|---|
| menor BIC (registro) | 11 | 72.844,9 | 72.719,0 | 1,035 |
| todos os subconjuntos (triagem quase-binomial, 200 refeitos; não exaustiva no beta-binomial), melhor BIC | 13 | 72.848,8 | 72.709,7 | 1,000 |
| menor AIC (backward, forward, both, HL, todos_aic) | 15 | 72.855,0 | 72.702,6 | 1,003 |
| cheio | 17 | 72.871,1 | 72.705,4 | 1,012 |

- Os erros por UF diferem por no máximo 0,035 p.p., dentro do ruído que a validação cruzada do notebook 03 mede melhor; por isso a regra de registro usa o BIC (parcimônia) e não o MAE. Ficar com o BIC custa 0,035 p.p. em relação ao melhor MAE (13 variáveis), e o notebook 03 decide se compensa.
- O modelo `bb_selecao` fica 607 pontos de BIC acima do `bb_uf` (Parte 9), isto é, a UF fixa explica muito mais do que qualquer seleção de contínuas.

## Parte 9 - Comparação

Todas as linhas são **dentro da amostra**. AIC/BIC só valem para modelos com verossimilhança binomial sobre (y, n): binomial, beta-binomial e (aproximadamente, sem penalização) L1.
Quase-binomial e GLMM-PQL não têm verossimilhança (QAIC = −2ℓ/φ + 2k aparece só para o quase-binomial); regressão beta, OLS e WLS têm outra resposta, então AIC/BIC ficam `NaN`.
O erro MAE ponderado (p.p.) dentro da amostra **favorece modelos mais flexíveis** e não substitui a validação cruzada.

In [17]:
for nome in E.CANDIDATOS:
    if nome not in AJ: ajustar(nome)
comp = E.tabela_comparacao(LINHAS)
comp.to_csv(TAB / "comparacao_modelos.csv", index=False)
print(comp[["modelo", "familia", "ligacao", "k", "aic", "bic", "qaic", "deviance_gl", "phi_pearson", "mae_pond_pp"]].to_string(index=False))

      modelo  familia    ligacao  k          aic          bic       qaic  deviance_gl  phi_pearson  mae_pond_pp
       bb_uf  betabin      logit 48  71,919.4948  72,237.5020        NaN      22.3506      22.3498       1.1664
   bb_reglag  betabin      logit 30  72,656.0457  72,854.8002        NaN      23.6296      23.6728       1.1434
   bb_spline  betabin      logit 29  72,679.8138  72,871.9431        NaN      23.8452      23.8697       1.1657
    bb_logit  betabin      logit 26  72,707.4286  72,879.6825        NaN      23.5916      23.6112       1.1548
  bb_selecao  betabin      logit 19  72,718.9958  72,844.8737        NaN      23.7632      23.7885       1.1468
   bb_offset  betabin      logit 25  73,480.0349  73,645.6637        NaN      30.1267      30.3014       1.3498
   bin_logit binomial      logit 25 172,593.2932 172,758.9220        NaN      21.7527      21.7714       1.0630
  bin_probit binomial     probit 25 176,281.6598 176,447.2886        NaN      22.4179      22.4614      

In [18]:
bbs = comp[comp.familia == "betabin"].set_index("modelo")
print((bbs[["aic", "bic"]] - bbs[["aic", "bic"]].min()).round(1).sort_values("bic").to_string())
print(f"tempo total do notebook: {time.time()-T0:.0f}s")

                  aic        bic
modelo                          
bb_uf          0.0000     0.0000
bb_selecao   799.5000   607.4000
bb_reglag    736.6000   617.3000
bb_spline    760.3000   634.4000
bb_logit     787.9000   642.2000
bb_offset  1,560.5000 1,408.2000
tempo total do notebook: 1741s


### Candidatos para a validação cruzada (notebook 03)

Não escolhemos o modelo final aqui. Os números acima são dentro da amostra; o notebook 03 decide por validação cruzada (k-fold e, para as previsões por UF, leave-one-UF-out). Levamos **cinco candidatos**, mais uma linha de base:

| Chave em `CANDIDATOS` | Por que entra |
|---|---|
| `bb_reglag` | Menor AIC entre os beta-binomiais sem efeito de UF; a inclinação do lag varia por região (LRT p ≈ 1e-11). Verossimilhança completa (permite intervalos). |
| `bb_selecao` | Menor BIC beta-binomial entre os métodos de seleção (11 contínuas além do `lag`, com Nordeste + Norte fundidos; BIC = 72.844,9): a opção parcimoniosa. Registrar com `E.registrar_selecao(vs, FUNDIR)` usando `selecao_variaveis.json` antes de chamar. |
| `bb_uf` | Menor AIC/BIC de todos (UF fixa melhora ≈ 770 de AIC sobre a região). Só serve em k-fold: no leave-one-UF-out o nível do estado de teste não existe e as dummies viram a referência (SP). |
| `quasi_reglag` | Pesa os municípios por votos (como o erro nacional), com o menor erro ponderado dentro da amostra entre os modelos sem UF (1,05 p.p.), e φ ≈ 21,5 corrige os EPs. Sem AIC; a CV decide. |
| `glmm_uf` | Intercepto aleatório por UF (σ ≈ 0,05 no logit): encolhe estados pequenos e aceita UF nova (efeito 0). É o candidato com UF válido no leave-one-UF-out. |

Linha de base: `bb_logit` (spec principal) e `quasi_logit`. Descartados: ligações probit/cloglog/loglog/cauchit (AIC muito pior que o logit), `*_offset` (offset rejeitado; ver abaixo), regressão beta e OLS/WLS (outra resposta, não ponderam por votos como queremos), `l1_logit` (o laço do fractional logit não pondera por votos e perde muito de ajuste).

**Conclusões da Parte 8 e 9.** (i) O offset do lag (swing puro) é rejeitado: o coeficiente livre do lag é ≈ 0,90 (z = −28 contra 1) e o AIC piora ≈ 750. Os municípios regridem à média em relação a 2022. (ii) A spline do lag e a interação região × lag melhoram a verossimilhança de modo significativo, mas pouco no erro ponderado. (iii) UF fixa melhora muito a verossimilhança; no erro ponderado, o quase-binomial com UF cai de 1,06 para 0,91 p.p. (iv) O beta-binomial dá θ ≈ 630 e ρ ≈ 0,0016: correlação intraclasse pequena, mas com nᵢ de milhares, 1 + (n−1)ρ é da ordem de 20 (a mesma sobredispersão do φ de Pearson no binomial). Os φ e a deviance/gl do beta-binomial são calculados com a variância binomial e **não** medem o ajuste dele: use o QQ-plot dos resíduos quantílicos (desvio ≈ 1,03 contra 3,4 no binomial).

## Parte 10 - Diagnóstico

Até aqui comparamos modelos por AIC/BIC dentro da amostra. Esta parte verifica se os **pressupostos** de cada candidato se sustentam, para que a escolha (feita por validação cruzada no notebook 03) não repouse em uma suposição falsa. Modelos diagnosticados: `bin_logit`, `quasi_reglag`, `bb_logit`, `bb_reglag`, `bb_spline` e `glmm_uf`.

Pressupostos e como cada verificação os testa:

1. **Distribuição da resposta** (binomial, ou beta-binomial): resíduos quantílicos aleatorizados (Dunn & Smyth, 1996) devem ser N(0,1); QQ-plot e gráfico half-normal com envelope simulado.
2. **Ausência de pontos que dominam o ajuste**: alavanca e distância de Cook; reajuste sem os 10 municípios mais influentes.
3. **Forma do preditor e variância**: resíduo contra o ajustado, contra `lag` (resultado de 2022) e por região.
4. **Independência entre municípios**: I de Moran dos resíduos (999 permutações) e mapa de resíduos.

**Resíduos quantílicos.** Para cada município, sorteamos u ~ U(F(y-1), F(y)), em que F é a função de distribuição ajustada, e tomamos r = Φ⁻¹(u). Para resposta discreta, o resíduo de Pearson ou o de deviance não é normal nem com o modelo certo; o quantílico é. Para a binomial usamos F binomial com π̂ᵢ e nᵢ; para a beta-binomial, a beta-binomial com θ̂. Os modelos quase-binomial e GLMM não têm verossimilhança completa; para eles os resíduos são os da binomial com o π̂ᵢ do modelo. Logo, um desvio padrão acima de 1 nesses dois mede a sobredispersão que eles absorvem apenas nos erros-padrão (quase) ou no efeito de UF (GLMM).

### Teoria - Resíduos de um MLG

**Pearson:** $r_P=\dfrac{y_i-n_i\hat\pi_i}{\sqrt{n_i\hat\pi_i(1-\hat\pi_i)}}$; $\sum r_P^2=X^2$, base de $\hat\phi$. **Deviance:** $r_D=\mathrm{sinal}(y_i-n_i\hat\pi_i)\sqrt{d_i}$, com $d_i$ a contribuição do município $i$ a $D$ (de modo que $\sum r_D^2=D$). Ambos são aproximadamente normais só quando $n_i$ é grande e o modelo é correto; para a binomial com sobredispersão passam a ter desvio padrão $\approx\sqrt{\phi}$, o que serve de diagnóstico.

**Resíduos quantílicos (Dunn & Smyth).** $r_i=\Phi^{-1}(u_i)$, $u_i\sim U(F(y_i-1),F(y_i))$, são exatamente $N(0,1)$ se $F$ é a distribuição correta. Servem para qualquer família com função de distribuição explícita, inclusive a beta-binomial, onde não há resíduo de Pearson ou deviance padrão e a dispersão extra está na própria $F$.

In [19]:
t10 = time.time()
MODELOS_D = ["bin_logit", "quasi_reglag", "bb_logit", "bb_reglag", "bb_spline", "glmm_uf"]
for nome in MODELOS_D:
    if nome not in AJ: ajustar(nome)
PI, TH, R = {}, {}, {}
for nome in MODELOS_D:
    c = E.CANDIDATOS[nome]; PI[nome] = np.asarray(c.prever(AJ[nome], df, mapa), float)
    TH[nome] = AJ[nome].extra.get("theta") if c.familia == "betabin" else None
    R[nome] = G.residuos_quantilicos_rapido(y, n, PI[nome], theta=TH[nome], rng=0)
# conferência da quadratura contra a FDA exata da scipy (mesmos sorteios uniformes): 100 maiores municípios + 1 a cada 30
sub = np.unique(np.r_[np.argsort(-n)[:100], np.arange(0, N, 30)])
r_ex = M.residuos_quantilicos(y[sub], n[sub], PI["bb_reglag"][sub], theta=TH["bb_reglag"], rng=0)
r_ap = G.residuos_quantilicos_rapido(y[sub], n[sub], PI["bb_reglag"][sub], theta=TH["bb_reglag"], rng=0)
print(f"quadratura x scipy ({len(sub)} mun.): máx |dif| = {np.abs(r_ex - r_ap).max():.4f}; percentil 99 = {np.quantile(np.abs(r_ex - r_ap), .99):.5f}")
reg_s = df[mapa["regiao"]].astype(str).to_numpy()
resumo = pd.DataFrame({nome: {"media": R[nome].mean(), "dp": R[nome].std(ddof=1),
                              "assimetria": stats.skew(R[nome]), "curtose_exc": stats.kurtosis(R[nome]),
                              "pct_abs_gt2": 100 * np.mean(np.abs(R[nome]) > 2),
                              "pct_abs_gt3": 100 * np.mean(np.abs(R[nome]) > 3)} for nome in MODELOS_D}).T
resumo.index.name = "modelo"; resumo.to_csv(TAB / "diag_residuos_resumo.csv")
pd.DataFrame(R).assign(id_municipio=df["id_municipio"].to_numpy()).to_csv(TAB / "diag_residuos_quantilicos.csv", index=False)
print(resumo.round(3).to_string())
print("esperado sob N(0,1): dp 1, assimetria 0, curtose 0, |r|>2: 4,55%, |r|>3: 0,27%")

quadratura x scipy (283 mun.): máx |dif| = 1.4321; percentil 99 = 0.02972
               media     dp  assimetria  curtose_exc  pct_abs_gt2  pct_abs_gt3
modelo                                                                        
bin_logit    -0.0070 3.4470     -0.0300      -0.6230      55.6730      38.9050
quasi_reglag  0.0140 3.4400     -0.0420      -0.6170      55.1710      38.8330
bb_logit     -0.0070 1.0340     -0.4230       3.9530       4.9550       1.0590
bb_reglag    -0.0030 1.0330     -0.4230       3.8030       4.8470       1.0590
bb_spline    -0.0040 1.0330     -0.4450       3.7970       4.9370       1.1130
glmm_uf      -0.1150 3.2590      0.0390      -0.4480      52.1010      35.2600
esperado sob N(0,1): dp 1, assimetria 0, curtose 0, |r|>2: 4,55%, |r|>3: 0,27%


### Passo 1 - QQ-plot dos resíduos quantílicos

Se a distribuição da resposta estiver certa, os pontos ficam sobre a diagonal. Inclinação maior que 1 indica variância subestimada (sobredispersão); curvatura nas caudas indica caudas mais pesadas que o modelo. Cada curva usa no máximo 1.500 pontos (a figura completa está em `results/figures/diag_qq.html`).

In [20]:
fig = G.qq_normal(R, "QQ-plot dos resíduos quantílicos (N(0,1) esperado)")
G.salvar(fig, FIG / "diag_qq.html"); fig.show()

### Passo 2 - Half-normal com envelope simulado

Gráfico de |resíduo| ordenado contra quantis half-normal, com banda de 95% obtida por **bootstrap paramétrico**: simulamos a resposta sob o modelo ajustado, reajustamos o modelo inteiro, recalculamos os resíduos e tomamos os quantis 2,5% e 97,5% ponto a ponto (Atkinson, 1985; Moral et al., 2017). O envelope por reajuste leva em conta que os parâmetros foram estimados dos mesmos dados.

**Decisão de custo:** `bin_logit` usa B = 99 (cada reajuste leva ~0,04 s). `bb_reglag` usa **B = 49**, porque cada reajuste beta-binomial leva ~4 s; com 49 simulações os extremos da banda são o 2º menor e o 2º maior valor simulados (nível nominal ~96%) e a banda é mais ruidosa. Pontos fora da banda são contados ponto a ponto, então uma fração de alguns por cento fora já é esperada mesmo com o modelo certo.

Os resíduos beta-binomiais usam uma quadratura da FDA (`G.residuos_quantilicos_rapido`): a `scipy.stats.betabinom.cdf` leva ~9 s por chamada, o que tornaria cada reajuste do envelope três vezes mais lento. A conferência no passo 1 mostra que 99% dos resíduos diferem menos de 0,04 da versão exata; as diferenças maiores ficam nas caudas extremas (|r| > 4), onde um erro de 1e-4 na FDA se amplifica em Φ⁻¹, e não alteram nenhuma conclusão.

In [21]:
# os resíduos de cada reajuste usam o y simulado daquele reajuste (estado['y'])
def env_modelo(nome, B, seed):
    c = E.CANDIDATOS[nome]; col_y = mapa["y"]; estado = {"y": y}
    def ajustar_fn(ys=None):
        estado["y"] = y if ys is None else ys
        return c.ajustar(df if ys is None else df.assign(**{col_y: ys}), mapa)
    def residuos_fn(res):
        return G.residuos_quantilicos_rapido(estado["y"], n, c.prever(res, df, mapa),
                                      theta=res.extra.get("theta") if c.familia == "betabin" else None, rng=0)
    def simular_fn(res, rng):
        pi_ = c.prever(res, df, mapa)
        if c.familia == "betabin":
            th = res.extra["theta"]; pi_ = rng.beta(pi_ * th, (1 - pi_) * th)
        return rng.binomial(n.astype(int), pi_).astype(float)
    return M.envelope_meio_normal(residuos_fn, simular_fn, ajustar_fn, B=B, rng=seed)

ENV, lin = {}, []
for nome, B in [("bin_logit", 99), ("bb_reglag", 49)]:
    t0 = time.time(); ENV[nome] = env_modelo(nome, B, 2026); e = ENV[nome]
    fora = (e["observado"] > e["superior"]) | (e["observado"] < e["inferior"])
    acima = e["observado"] > e["superior"]
    lin.append({"modelo": nome, "B": B, "pct_fora": 100 * fora.mean(), "pct_acima": 100 * acima.mean(), "pct_abaixo": 100 * (e["observado"] < e["inferior"]).mean(),
                "maior_razao_obs_sup": float((e["observado"] / e["superior"]).max()), "segundos": time.time() - t0})
    pd.DataFrame({k: e[k] for k in ["teorico", "observado", "inferior", "mediana", "superior"]}).to_csv(TAB / f"diag_envelope_{nome}.csv", index=False)
env_res = pd.DataFrame(lin)
# o tempo de execução varia a cada rodada: fica na saída, não na tabela versionada
env_res.drop(columns="segundos").to_csv(TAB / "diag_envelope_resumo.csv", index=False)
print(env_res.round(3).to_string())
fig = G.meio_normal(ENV, "Half-normal com envelope simulado de 95% (bin_logit B=99; bb_reglag B=49)")
G.salvar(fig, FIG / "diag_meio_normal.html"); fig.show()

      modelo   B  pct_fora  pct_acima  pct_abaixo  maior_razao_obs_sup  segundos
0  bin_logit  99  100.0000   100.0000      0.0000               4.0860    7.0080
1  bb_reglag  49   92.4780     5.1530     87.3250               1.7280  543.2300


### Passo 3 - Alavanca e distância de Cook

A alavanca e a distância de Cook estão definidas na Teoria abaixo. Usamos o `quasi_reglag` (GLM quase-binomial, φ̂ de Pearson). Calculamos à mão, porque o `get_influence()` do statsmodels não trata a resposta de duas colunas com φ de Pearson própria. Valores de referência convencionais: h > 2p/N e D > 4/N, que num N grande sinalizam muitos municípios e servem só como triagem; o que importa é o efeito do reajuste sem os 10 maiores.

Como os pesos da binomial crescem com nᵢ, os municípios grandes (capitais) tendem a ter alavanca alta.

### Teoria - Alavanca e Cook em um MLG

No MLG a matriz-chapéu é $H=W^{1/2}X(X^\top WX)^{-1}X^\top W^{1/2}$, com $W$ os pesos do IRLS. Ao contrário dos mínimos quadrados, **$H$ depende do ajuste** (de $\hat\pi_i$ e de $n_i$), não só de $X$: um município grande ou com $\hat\pi_i$ perto de 0,5 pesa mais. A distância de Cook generalizada é
$$D_i=\frac{r_{P,i}^2\,h_{ii}}{\hat\phi\,p\,(1-h_{ii})^2},$$
a variação aproximada de $\hat\beta$ ao retirar o município $i$, em unidades da sua região de confiança.

In [22]:
a_q = AJ["quasi_reglag"]; cq = E.CANDIDATOS["quasi_reglag"]
Xq = E.montar_X(df, mapa, a_q.pad, cq.termos).to_numpy(float); pq = Xq.shape[1]; phi_q = a_q.extra["phi"]
pi_q = PI["quasi_reglag"]; W = n * pi_q * (1 - pi_q)
A = np.linalg.inv(Xq.T @ (W[:, None] * Xq))
h = W * np.einsum("ij,jk,ik->i", Xq, A, Xq)
rP = (y - n * pi_q) / np.sqrt(W)
cook = rP ** 2 * h / (phi_q * pq * (1 - h) ** 2)
infl = df[["id_municipio", "nome_municipio", "sigla_uf"]].assign(n=n, prop_obs=y / n, prop_aj=pi_q, alavanca=h, cook=cook, pearson=rP)
top10 = infl.sort_values("cook", ascending=False).head(10)
top10.to_csv(TAB / "diag_influencia_top10.csv", index=False)
print(f"p = {pq}; phi = {phi_q:.2f}; alavanca média = {h.mean():.4f}; max = {h.max():.3f}; "
      f"h > 2p/N: {(h > 2*pq/N).sum()} mun.; D > 4/N: {(cook > 4/N).sum()} mun.; max D = {cook.max():.4f}")
print(top10.round(4).to_string(index=False))
fig = G.alavanca_cook(h, cook, (df["nome_municipio"].astype(str) + "/" + df["sigla_uf"].astype(str)).tolist(), pq)
G.salvar(fig, FIG / "diag_alavanca_cook.html"); fig.show()

p = 29; phi = 21.56; alavanca média = 0.0052; max = 0.672; h > 2p/N: 483 mun.; D > 4/N: 426 mun.; max D = 5.9917
 id_municipio   nome_municipio sigla_uf              n  prop_obs  prop_aj  alavanca   cook  pearson
      5300108         Brasília       DF 1,804,766.0000    0.4572   0.4482    0.6715 5.9917  24.5323
      3550308        São Paulo       SP 6,962,277.0000    0.5550   0.5608    0.5542 4.1475 -30.4887
      1302603           Manaus       AM 1,178,310.0000    0.4417   0.4250    0.2408 0.8907  36.5059
      2304400        Fortaleza       CE 1,551,916.0000    0.6007   0.6124    0.2637 0.6899 -29.7817
      2611606           Recife       PE 1,033,711.0000    0.6127   0.5968    0.1748 0.4454  32.9337
      5208707          Goiânia       GO   805,038.0000    0.3736   0.3874    0.1696 0.2536 -25.3813
      3304557   Rio de Janeiro       RJ 3,683,589.0000    0.5154   0.5124    0.3564 0.1760  11.3073
      4314902     Porto Alegre       RS   816,709.0000    0.5584   0.5507    0.2158 0.1

In [23]:
# Reajuste sem os 10 municípios mais influentes; mudança em unidades de EP do ajuste completo.
# A padronização (z-score) é refeita no subconjunto, o que desloca os coeficientes de forma desprezível (dp muda ~1e-4 em termos relativos).
df_s = df.drop(index=top10.index)
linhas = []
for nome in ["quasi_reglag", "bb_reglag"]:
    c = E.CANDIDATOS[nome]; a_s = c.ajustar(df_s, mapa)
    t_c, t_s = E.tabela_coeficientes(AJ[nome]), E.tabela_coeficientes(a_s)
    comum = [i for i in t_c.index if i in t_s.index]
    d = pd.DataFrame({"modelo": nome, "termo": comum, "coef_completo": t_c.loc[comum, "coef"].to_numpy(),
                      "coef_sem10": t_s.loc[comum, "coef"].to_numpy(), "ep_completo": t_c.loc[comum, "ep"].to_numpy()})
    d["mudanca_em_ep"] = (d.coef_sem10 - d.coef_completo) / d.ep_completo
    linhas.append(d)
sem10 = pd.concat(linhas, ignore_index=True); sem10.to_csv(TAB / "diag_sem_influentes.csv", index=False)
for nome, d in sem10.groupby("modelo"):
    j = d.mudanca_em_ep.abs().idxmax()
    print(f"{nome}: maior |mudança| = {d.mudanca_em_ep[j]:+.2f} EP em '{d.termo[j]}'; termos com |mudança| > 0,5 EP: {(d.mudanca_em_ep.abs() > 0.5).sum()} de {len(d)}; "
          f"termos que trocam de sinal: {(np.sign(d.coef_completo) != np.sign(d.coef_sem10)).sum()}")
print(sem10.assign(a=sem10.mudanca_em_ep.abs()).sort_values("a", ascending=False).head(6).drop(columns="a").round(4).to_string(index=False))

bb_reglag: maior |mudança| = -0.15 EP em 'bloco_outro'; termos com |mudança| > 0,5 EP: 0 de 30; termos que trocam de sinal: 0
quasi_reglag: maior |mudança| = -8.99 EP em 'bloco_outro'; termos com |mudança| > 0,5 EP: 21 de 29; termos que trocam de sinal: 3
      modelo       termo  coef_completo  coef_sem10  ep_completo  mudanca_em_ep
quasi_reglag bloco_outro         0.0806      0.0252       0.0062        -8.9920
quasi_reglag     reg_Sul        -0.0365     -0.0562       0.0062        -3.1979
quasi_reglag     log_pop         0.0057      0.0095       0.0012         3.1070
quasi_reglag reg_Sul:lag         0.0118     -0.0049       0.0058        -2.8666
quasi_reglag         lag         0.7725      0.7845       0.0042         2.8386
quasi_reglag   reg_Norte         0.0445      0.0281       0.0066        -2.4997


### Passo 4 - Resíduos contra ajustado, contra `lag` e por região

O resíduo quantílico deve ter média 0 e dispersão 1 em qualquer faixa do ajustado (η̂ = logit π̂), do `lag` e em cada região. A curva vermelha é a média do resíduo em 20 faixas com o mesmo número de municípios: tendência na curva indica forma funcional errada (por exemplo, o coeficiente do `lag` ou um termo faltando); funil indica variância mal especificada. As correlações de Spearman e o desvio padrão por região (tabelas abaixo) resumem os gráficos.

In [24]:
eta = {k: np.log(PI[k] / (1 - PI[k])) for k in MODELOS_D}
x_lag = df[mapa["lag"]].to_numpy(float)
fig1 = G.residuos_contra(eta["bb_logit"], R, "logit do ajustado (η̂ de bb_logit, mesmo eixo em todos)", "Resíduo contra o ajustado")
fig2 = G.residuos_contra(x_lag, R, "lag = logit PT 2º turno 2022", "Resíduo contra o lag")
fig3 = G.caixa_regiao(R, reg_s, "Resíduos quantílicos por região")
for f, nm in [(fig1, "res_ajustado"), (fig2, "res_lag"), (fig3, "res_regiao")]:
    G.salvar(f, FIG / f"diag_{nm}.html"); f.show()
corr = pd.DataFrame({k: {"spearman_com_eta": stats.spearmanr(R[k], eta[k])[0], "spearman_com_lag": stats.spearmanr(R[k], x_lag)[0],
                         "corr_abs_r_eta": stats.spearmanr(np.abs(R[k]), eta[k])[0]} for k in MODELOS_D}).T
dp_reg = pd.DataFrame({k: pd.Series(R[k]).groupby(reg_s).std() for k in MODELOS_D}).T
med_reg = pd.DataFrame({k: pd.Series(R[k]).groupby(reg_s).mean() for k in MODELOS_D}).T
corr.rename_axis("modelo").to_csv(TAB / "diag_correlacoes.csv"); dp_reg.rename_axis("modelo").to_csv(TAB / "diag_dp_regiao.csv")
med_reg.rename_axis("modelo").to_csv(TAB / "diag_media_regiao.csv")
print(corr.round(3).to_string()); print("DP do resíduo por região:"); print(dp_reg.round(2).to_string())
print("média do resíduo por região:"); print(med_reg.round(2).to_string())

              spearman_com_eta  spearman_com_lag  corr_abs_r_eta
bin_logit               0.0200            0.0270          0.0980
quasi_reglag            0.0300            0.0350          0.0920
bb_logit                0.0120            0.0150          0.0980
bb_reglag               0.0140            0.0160          0.0900
bb_spline               0.0080            0.0100          0.0940
glmm_uf                 0.0020            0.0040          0.0870
DP do resíduo por região:
              Centro-Oeste  Nordeste  Norte  Sudeste    Sul
bin_logit           3.4900    3.8700 4.0000   3.0800 2.9300
quasi_reglag        3.5400    3.8600 3.9400   3.0700 2.9300
bb_logit            0.9600    1.1900 1.4300   0.8000 0.9200
bb_reglag           0.9600    1.1900 1.4000   0.8000 0.9300
bb_spline           0.9600    1.1800 1.4200   0.8000 0.9300
glmm_uf             3.1300    3.6100 3.8500   2.9600 2.8100
média do resíduo por região:
              Centro-Oeste  Nordeste   Norte  Sudeste     Sul
bin_logi

### Passo 5 - Autocorrelação espacial dos resíduos

O pressuposto de independência entre municípios é o mais frágil: a EDA achou I de Moran = 0,896 para a proporção do PT e 0,542 para o *swing*. Calculamos o I de Moran dos resíduos quantílicos de cada modelo (esda, pesos de contiguidade rainha padronizados por linha, municípios-ilha ligados ao vizinho mais próximo, como no notebook 01; 999 permutações). Se os preditores absorvem a estrutura espacial, I ≈ 0. Se I > 0 permanece, municípios vizinhos erram na mesma direção; a informação efetiva é menor que N e **os erros-padrão e intervalos continuam pequenos demais** mesmo no beta-binomial. A estimativa pontual não fica necessariamente enviesada, mas o teste de significância e os intervalos ficam otimistas. A checagem honesta de desempenho é a validação cruzada *deixando uma UF de fora por vez* (notebook 03), que não deixa vizinhos no treino em que o teste foi avaliado, exceto na fronteira entre UFs.

O mapa de resíduos de `bb_reglag` é salvo só em HTML (`results/figures/diag_mapa_residuos_bb_reglag.html`, geometria simplificada a 0,05°), para não pesar o notebook.

In [25]:
geo = pd.read_parquet(ROOT / "data/processed/municipios_geo.parquet")
geo["id_municipio"] = geo["id_municipio"].astype(df["id_municipio"].dtype)
geo = geo.set_index("id_municipio").loc[df["id_municipio"]].reset_index()
gdf = gpd.GeoDataFrame(geo, geometry=gpd.GeoSeries.from_wkb(geo["geometry"]), crs="EPSG:4674")
w = Queen.from_dataframe(gdf, use_index=False)
w = attach_islands(w, KNN.from_dataframe(gdf, k=1)); w.transform = "r"
print("ilhas após ligação:", len(w.islands), "| vizinhos médios:", round(w.mean_neighbors, 2))
linhas = []
for k in MODELOS_D + ["bin_logit_pearson"]:
    r_k = R[k] if k in R else (y - n * PI["bin_logit"]) / np.sqrt(n * PI["bin_logit"] * (1 - PI["bin_logit"]))
    np.random.seed(0)  # permutações reprodutíveis
    mi = esda.Moran(r_k, w, permutations=999)
    linhas.append({"modelo": k, "I": mi.I, "esperado": mi.EI, "z_sim": mi.z_sim, "p_sim": mi.p_sim, "dp_residuo": r_k.std(ddof=1)})
moran = pd.DataFrame(linhas); moran.to_csv(TAB / "diag_moran.csv", index=False)
print(moran.round(4).to_string(index=False))
fig = G.mapa_residuos(gdf, R["bb_reglag"], "Resíduos quantílicos de bb_reglag (limitados a ±4)")
G.salvar(fig, FIG / "diag_mapa_residuos_bb_reglag.html")
print("mapa salvo; tempo da Parte 10:", round(time.time() - t10), "s")

ilhas após ligação: 0 | vizinhos médios: 5.79


           modelo      I  esperado   z_sim  p_sim  dp_residuo
        bin_logit 0.3513   -0.0002 44.3473 0.0010      3.4466
     quasi_reglag 0.3563   -0.0002 45.1238 0.0010      3.4402
         bb_logit 0.3245   -0.0002 40.9516 0.0010      1.0341
        bb_reglag 0.3294   -0.0002 41.3105 0.0010      1.0328
        bb_spline 0.3234   -0.0002 40.8971 0.0010      1.0330
          glmm_uf 0.2878   -0.0002 36.3946 0.0010      3.2593
bin_logit_pearson 0.3426   -0.0002 43.1187 0.0010      4.6553


mapa salvo; tempo da Parte 10: 603 s


### Passo 6 - Pressupostos dos modelos escolhidos (`bb_selecao` e `bb_slx`)

Os Passos 1 a 5 diagnosticam os modelos antigos. Aqui os dois modelos que a Parte 8 e a Parte 14 escolhem passam pela bateria completa: `bb_selecao` (menor BIC pela regra de 1 EP) e `bb_slx` (modelo final: dispersão `regiao_logn`, defasagens espaciais SLX das 12 covariáveis e Nordeste + Norte fundidos na média). **Dependência para a frente:** o `bb_slx` é montado a partir de `results/tables/final_decisao.json`, saída do notebook 05; se o arquivo não existir, só o `bb_selecao` é diagnosticado e as seções do `bb_slx` são puladas com uma mensagem. Os testes são diagnóstico: nenhum modelo muda aqui e qualquer remédio vira pendência.

As regras de decisão foram fixadas antes de ver os números:

1. **Ligação (Pregibon/Hinkley).** Reajusta-se o modelo com $\hat\eta^2$ (padronizado) como covariável extra na média. Wald do coeficiente com a covariância de Conley (corte da Parte 15) e, para comparar, com a covariância do modelo. A ligação é sinalizada se $p_{\text{Conley}} < 0{,}05$; o tamanho do efeito é o percentil 95 e o máximo de $|\Delta\hat\pi|$ em pontos percentuais.
2. **Linearidade.** Para cada covariável contínua da média (as 11 + `lag`; as `w_*` ficam fora): resíduo parcial $r^{trab}_i + \hat\beta_j x_{ij}$ contra $x_{ij}$ em 20 faixas de quantil (médias ponderadas por $n$) e troca de $x_j$ por spline cúbica natural de 4 gl. A troca é testada acrescentando à média os 3 termos não lineares (ortogonais a $1$ e $x_j$), que gera o mesmo modelo; Wald de 3 gl com a covariância de Conley. **Não linearidade material** se $p_{\text{Conley}} < 0{,}05$ **e** o percentil 95 de $|\Delta\hat\pi| \geq 0{,}5$ p.p.
3. **Distribuição.** Resíduos quantílicos aleatorizados (`rng=0`) com $\theta_i$ por município, QQ e envelope meio-normal (mesmo procedimento do Passo 2), comparados ao `bb_reglag` antigo.
4. **Variância.** DP dos resíduos por região e por quintil de $n$; Spearman de $|r|$ com $\hat\eta$ e com $\log n$.
5. **Independência.** I de Moran (999 permutações, pesos rainha do Passo 5). O erro-padrão de Conley corrige a inferência (Parte 15); nada além disso é feito aqui.
6. **Colinearidade.** VIF da matriz de média padronizada do `bb_slx` (com as `w_*`); para fatores, $\mathrm{GVIF}^{1/(2\,gl)}$. VIF > 10 sinaliza que a leitura de coeficientes individuais fica frágil; a previsão não é afetada.
7. **Influência (beta-binomial).** DFBETA de um passo $\Delta\beta_i \approx H^{-1}s_i$ e Cook $=\Delta\beta_i'(-H)\Delta\beta_i/p$ ($s_i$ = escore da observação, $H$ = hessiana da log-verossimilhança). Reajuste sem os 10 maiores; mudança em EP de Conley.

In [26]:
t6 = time.time()
sel_json = TAB / "selecao_variaveis.json"
dec_json = TAB / "final_decisao.json"
CORTE_KM = float(json.loads(dec_json.read_text())["corte_final"]) if dec_json.exists() else 200.0
W6 = S.pesos_queen(gdf)
lat, lon = df["lat"].to_numpy(float), df["lon"].to_numpy(float)
MS = {}   # modelo -> dict(res, df, mapa, cand, aj, pi, theta)
def registrar_ms(nome, cand, df_, mapa_):
    t0 = time.time(); a = cand.ajustar(df_, mapa_)
    MS[nome] = {"cand": cand, "aj": a, "res": a.modelo, "df": df_, "mapa": mapa_,
                "pi": np.asarray(cand.prever(a, df_, mapa_), float), "theta": E.prever_theta(a, df_, mapa_),
                "seg_ajuste": time.time() - t0}
    print(f"{nome}: {len(a.modelo.params)} parâmetros | {MS[nome]['seg_ajuste']:.1f} s por ajuste | phi de Pearson = "
          f"{np.sum((y - n * MS[nome]['pi']) ** 2 / (n * MS[nome]['pi'] * (1 - MS[nome]['pi']))) / (N - len(a.modelo.params)):.1f} | theta mediano = {np.median(MS[nome]['theta']):.0f}")
registrar_ms("bb_selecao", E.CANDIDATOS["bb_selecao"], df, mapa)
if dec_json.exists():
    tf = json.loads(dec_json.read_text())["termos_final"]
    CF6 = Candidato("bb_slx", "betabin", "logit", Termos(vars=tuple(v for v in tf["vars"] if not v.startswith("w_")), slx=tuple(tf["slx"]),
                    disp=tf["disp"], regiao_fundir=tuple(tuple(g) for g in tf["regiao_fundir"])))
    df_slx, mapa_slx = S.adicionar_slx(df, mapa, list(CF6.termos.slx), W6)
    registrar_ms("bb_slx", CF6, df_slx, mapa_slx)
else:
    print("results/tables/final_decisao.json não existe (saída do notebook 05): só bb_selecao entra neste passo.")
print("corte de Conley:", CORTE_KM, "km")

bb_selecao: 19 parâmetros | 2.8 s por ajuste | phi de Pearson = 23.8 | theta mediano = 620


bb_slx: 36 parâmetros | 7.4 s por ajuste | phi de Pearson = 23.3 | theta mediano = 731
corte de Conley: 200.0 km


#### 6.1 · Ligação

Regra: sinaliza se $p_{\text{Conley}} < 0{,}05$ no coeficiente de $\hat\eta^2$; o efeito é medido em $|\Delta\hat\pi|$ (p.p.).

In [27]:
# Ligação (Pregibon/Hinkley): η̂² padronizado como covariável extra na média
linhas = []
for nome, m in MS.items():
    t0 = time.time(); r_l = PR.teste_ligacao(m["res"], lat, lon, CORTE_KM)
    linhas.append({"modelo": nome, **r_l, "sinaliza": r_l["p_conley"] < 0.05, "seg": time.time() - t0})
LIG = pd.DataFrame(linhas); LIG.drop(columns="seg").to_csv(TAB / "diag_final_ligacao.csv", index=False)
print(LIG.round(4).to_string(index=False))

    modelo    coef  p_conley  p_modelo  dpi_p95_pp  dpi_max_pp  dloglik  sinaliza    seg
bb_selecao -0.0068    0.0277    0.0000      0.2141      0.4440  10.4115      True 4.9616
    bb_slx -0.0103    0.0003    0.0000      0.3288      0.5873  23.7129      True 7.6423


**Leitura.** `bb_selecao`: coeficiente de $\hat\eta^2$ = -0,0068, $p_{\text{Conley}}$ = 0,028 (covariância do modelo: < 0,001); $|\Delta\hat\pi|$ percentil 95 = 0,21 p.p., máximo = 0,44 p.p.; ganho de log-verossimilhança = 10,4. `bb_slx`: coeficiente de $\hat\eta^2$ = -0,0103, $p_{\text{Conley}}$ = < 0,001 (covariância do modelo: < 0,001); $|\Delta\hat\pi|$ percentil 95 = 0,33 p.p., máximo = 0,59 p.p.; ganho de log-verossimilhança = 23,7. **Veredito pela regra:** ligação sinalizada em `bb_selecao`, `bb_slx`. A covariância do modelo rejeita sempre com folga maior que a de Conley, o que repete o padrão da Parte 15 (erro-padrão ingênuo pequeno demais). O desvio da ligação logit é, porém, de tamanho pequeno nas previsões: mudar a curvatura mexe no máximo 0,59 p.p. em um município e 0,33 p.p. no percentil 95. Pendência (não aplicada): comparar ligações alternativas (probit, cloglog) no `bb_slx` apenas se a validação cruzada indicar ganho.

#### 6.2 · Linearidade de cada covariável contínua

Regra: não linearidade material se $p_{\text{Conley}} < 0{,}05$ e percentil 95 de $|\Delta\hat\pi| \geq 0{,}5$ p.p. Figuras: `results/figures/diag_final_parcial_<modelo>.html`.

In [28]:
# Linearidade: resíduo parcial em 20 faixas e spline natural de 4 gl, uma covariável por vez
linhas, FIG_PARCIAL = [], {}
for nome, m in MS.items():
    res, nomes = m["res"], list(m["aj"].nomes)
    cont = [(j, c) for j, c in enumerate(nomes[:res.model.exog.shape[1]]) if c in m["cand"].termos.vars and not c.startswith("w_")]
    fig = make_subplots(rows=4, cols=3, subplot_titles=[c for _, c in cont], vertical_spacing=0.08)
    for k, (j, c) in enumerate(cont):
        rp = PR.residuo_parcial(res, j); xj = res.model.exog[:, j]
        mf = PR.medias_por_faixa(xj, rp, n, 20)
        fig.add_scatter(x=mf["x"], y=mf["v"], mode="lines+markers", line=dict(color="crimson"), showlegend=False, row=k // 3 + 1, col=k % 3 + 1)
        fig.add_scatter(x=[xj.min(), xj.max()], y=float(res.params[j]) * np.array([xj.min(), xj.max()]), mode="lines",
                        line=dict(color="gray", dash="dash"), showlegend=False, row=k // 3 + 1, col=k % 3 + 1)
        t0 = time.time(); r_s = PR.teste_spline(res, j, lat, lon, CORTE_KM, gl=4)
        # desvio máximo da média por faixa em relação à reta, na escala do preditor linear (logit)
        dev = float(np.abs(mf["v"] - float(res.params[j]) * mf["x"]).max())
        linhas.append({"modelo": nome, "covariavel": c, "beta": float(res.params[j]), **r_s, "dev_max_faixa": dev,
                       "nao_linear": bool(r_s["p_conley"] < 0.05 and r_s["dpi_p95_pp"] >= 0.5)})
    fig.update_layout(title=f"Resíduo parcial por faixa de quantil (vermelho) e reta ajustada (cinza): {nome}", template="plotly_white", height=900)
    G.salvar(fig, FIG / f"diag_final_parcial_{nome}.html"); FIG_PARCIAL[nome] = fig
LINEAR = pd.DataFrame(linhas); LINEAR.to_csv(TAB / "diag_final_linearidade.csv", index=False)
print(LINEAR.drop(columns=["chi2_conley", "dloglik"]).round(4).to_string(index=False))
print("não lineares (regra: p_Conley < 0,05 e p95 |Δπ̂| ≥ 0,5 p.p.):", LINEAR.loc[LINEAR.nao_linear, ["modelo", "covariavel"]].values.tolist())

    modelo        covariavel    beta  p_conley  p_modelo  dpi_p95_pp  dpi_max_pp  dev_max_faixa  nao_linear
bb_selecao               lag  0.7726    0.0057    0.0000      0.2737      0.4391         0.0260       False
bb_selecao       pct_dep_esq  0.0106    0.0007    0.0000      0.2896      0.8410         0.0204       False
bb_selecao    comparecimento -0.0134    0.5596    0.1803      0.1109      0.5480         0.0216       False
bb_selecao          log_dens  0.0351    0.0027    0.0000      0.3477      1.1550         0.0285       False
bb_selecao      pct_mulheres  0.0287    0.1337    0.0545      0.1283      0.3033         0.0177       False
bb_selecao         pct_16_24 -0.0103    0.1322    0.0039      0.1902      0.6548         0.0332       False
bb_selecao      pct_superior  0.0063    0.0051    0.0000      0.3136      0.5891         0.0299       False
bb_selecao    pct_analfabeto -0.0072    0.0158    0.0000      0.2954      0.5785         0.0247       False
bb_selecao     cobertura_pbf

**Leitura.** `bb_selecao`: 12 covariáveis testadas; 8 com $p_{\text{Conley}} < 0{,}05$ no teste de spline, mas só 1 atende também ao critério de tamanho (`pct_pretos_pardos`: $p_{\text{Conley}}$ = < 0,001, $|\Delta\hat\pi|$ percentil 95 = 0,66 p.p., máximo = 1,53 p.p.). O maior percentil 95 entre as demais é 0,49 p.p. (cobertura_pbf). `bb_slx`: 12 covariáveis testadas; 11 com $p_{\text{Conley}} < 0{,}05$ no teste de spline, mas só 1 atende também ao critério de tamanho (`pct_pretos_pardos`: $p_{\text{Conley}}$ = < 0,001, $|\Delta\hat\pi|$ percentil 95 = 0,65 p.p., máximo = 1,64 p.p.). O maior percentil 95 entre as demais é 0,49 p.p. (cobertura_pbf). No resíduo parcial, o maior afastamento da média de uma faixa em relação à reta ajustada é 0,047 (escala do logit), em `pct_16_24` do `bb_slx`. **Veredito pela regra:** a forma linear no logit se sustenta para todas as covariáveis, exceto `pct_pretos_pardos`, onde há curvatura material nos dois modelos. Pendência (não aplicada): spline em `pct_pretos_pardos`, a validar por validação cruzada; os testes nominais com $p_{\text{Conley}} < 0{,}05$ sem efeito material não pedem mudança.

#### 6.3 · Distribuição: resíduos quantílicos, QQ e envelope

O `bb_slx` usa $\theta_i$ por município (`E.prever_theta`). O envelope usa B = 49 reajustes (cada ajuste leva poucos segundos; o limite fixado para trocar por B = 39 era 15 s por ajuste) e é comparado ao `bb_reglag` do Passo 2.

In [29]:
# Distribuição: resíduos quantílicos aleatorizados (rng=0), θᵢ por município
RF = {nome: G.residuos_quantilicos_rapido(y, n, m["pi"], theta=m["theta"], rng=0) for nome, m in MS.items()}
resumo_f = pd.DataFrame({nome: {"media": r_.mean(), "dp": r_.std(ddof=1), "assimetria": stats.skew(r_), "curtose_exc": stats.kurtosis(r_),
                                "pct_abs_gt2": 100 * np.mean(np.abs(r_) > 2), "pct_abs_gt3": 100 * np.mean(np.abs(r_) > 3)} for nome, r_ in RF.items()}).T
resumo_f.index.name = "modelo"; resumo_f.to_csv(TAB / "diag_final_residuos_resumo.csv")
pd.DataFrame(RF).assign(id_municipio=df["id_municipio"].to_numpy()).to_csv(TAB / "diag_final_residuos_quantilicos.csv", index=False)
print(pd.concat([resumo.loc[["bb_reglag"]], resumo_f]).round(3).to_string())
print("esperado sob N(0,1): dp 1, assimetria 0, curtose 0, |r|>2: 4,55%, |r|>3: 0,27%")
fig = G.qq_normal(RF, "QQ-plot dos resíduos quantílicos: bb_selecao e bb_slx")
G.salvar(fig, FIG / "diag_final_qq.html"); fig.show()

             media     dp  assimetria  curtose_exc  pct_abs_gt2  pct_abs_gt3
modelo                                                                      
bb_reglag  -0.0030 1.0330     -0.4230       3.8030       4.8470       1.0590
bb_selecao -0.0080 1.0340     -0.4400       3.9480       4.8830       1.0590
bb_slx     -0.0180 1.0250     -0.3840       2.4870       5.0630       0.8800
esperado sob N(0,1): dp 1, assimetria 0, curtose 0, |r|>2: 4,55%, |r|>3: 0,27%


In [30]:
# Envelope meio-normal por bootstrap paramétrico (mesmo procedimento do Passo 2, com θᵢ por município)
def env_modelo_f(m, B, seed):
    c, df_, mapa_ = m["cand"], m["df"], m["mapa"]; estado = {"y": y}
    def ajustar_fn(ys=None):
        estado["y"] = y if ys is None else ys
        return c.ajustar(df_ if ys is None else df_.assign(**{mapa_["y"]: ys}), mapa_)
    def residuos_fn(a):
        return G.residuos_quantilicos_rapido(estado["y"], n, c.prever(a, df_, mapa_), theta=E.prever_theta(a, df_, mapa_), rng=0)
    def simular_fn(a, rng):
        pi_ = c.prever(a, df_, mapa_); th = E.prever_theta(a, df_, mapa_)
        return rng.binomial(n.astype(int), rng.beta(pi_ * th, (1 - pi_) * th)).astype(float)
    return M.envelope_meio_normal(residuos_fn, simular_fn, ajustar_fn, B=B, rng=seed)
B_ENV = 49 if max(m["seg_ajuste"] for m in MS.values()) <= 15 else 39
print("B do envelope:", B_ENV, "(tempo máximo de um ajuste:", round(max(m["seg_ajuste"] for m in MS.values()), 1), "s)")
ENV_F, linhas = {}, []
for nome, m in MS.items():
    t0 = time.time(); ENV_F[nome] = env_modelo_f(m, B_ENV, 2026); e = ENV_F[nome]
    acima, abaixo = e["observado"] > e["superior"], e["observado"] < e["inferior"]
    linhas.append({"modelo": nome, "B": B_ENV, "pct_fora": 100 * (acima | abaixo).mean(), "pct_acima": 100 * acima.mean(), "pct_abaixo": 100 * abaixo.mean(),
                   "maior_razao_obs_sup": float((e["observado"] / e["superior"]).max()), "segundos": time.time() - t0})
    pd.DataFrame({k: e[k] for k in ["teorico", "observado", "inferior", "mediana", "superior"]}).to_csv(TAB / f"diag_final_envelope_{nome}.csv", index=False)
env_f = pd.DataFrame(linhas); env_f.drop(columns="segundos").to_csv(TAB / "diag_final_envelope_resumo.csv", index=False)
print(pd.concat([env_res[env_res.modelo == "bb_reglag"], env_f]).round(3).to_string(index=False))
fig = G.meio_normal(ENV_F, f"Half-normal com envelope simulado de 95% (B={B_ENV}): bb_selecao e bb_slx")
G.salvar(fig, FIG / "diag_final_meio_normal.html"); fig.show()

B do envelope: 49 (tempo máximo de um ajuste: 7.4 s)


    modelo  B  pct_fora  pct_acima  pct_abaixo  maior_razao_obs_sup  segundos
 bb_reglag 49   92.4780     5.1530     87.3250               1.7280  543.2300
bb_selecao 49   90.2150     4.8470     85.3680               1.7310  413.4940
    bb_slx 49   84.2190     5.6910     78.5280               1.4600  564.1980


**Leitura.** Resíduos quantílicos (esperado N(0,1)): `bb_reglag` antigo: DP 1,033, assimetria -0,42, curtose em excesso 3,80, 4,85% fora de ±2, 1,06% fora de ±3. `bb_selecao`: DP 1,034, assimetria -0,44, curtose 3,95, 4,88% e 1,06%. `bb_slx`: DP 1,025, assimetria -0,38, curtose 2,49, 5,06% e 0,88%. Envelope meio-normal (B = 49): `bb_reglag` antigo (B = 49): 92,5% fora, 5,15% acima, 87,3% abaixo. `bb_selecao`: 90,2% fora, 4,85% acima, 85,4% abaixo (maior razão observado/limite superior 1,73). `bb_slx`: 84,2% fora, 5,69% acima, 78,5% abaixo (maior razão observado/limite superior 1,46). **Pergunta: a dispersão por região corrige o padrão "corpo abaixo, caudas acima"?** Só em parte. A escala ficou certa (DP próximo de 1) e a curtose em excesso caiu de 3,80 (`bb_reglag`) para 2,49 (`bb_slx`); a fração fora do envelope caiu de 92,5% para 84,2% e a abaixo da banda de 87,3% para 78,5%. Mas a fração acima da banda não cai (5,15% para 5,69%) e a maior razão observado/limite superior continua em 1,46: o corpo continua estreito e as caudas pesadas demais. **Veredito pela regra:** a escala se sustenta, o formato da distribuição não. O `bb_slx` melhora o formato frente ao `bb_selecao`, sem resolvê-lo. Pendência (não aplicada): família de cauda mais pesada ou mistura de dispersão.

#### 6.4 · Variância dos resíduos

DP dos resíduos quantílicos por região e por quintil de $n$, e correlação de Spearman de $|r|$ com $\hat\eta$ e com $\log n$.

In [31]:
# Variância: DP dos resíduos por região e por quintil de n; Spearman de |r| com η̂ e com log n
linhas = []
q_n = pd.qcut(n, 5, labels=[f"Q{i}" for i in range(1, 6)]).astype(str)
for nome, m in MS.items():
    r_ = RF[nome]; eta_ = np.log(m["pi"] / (1 - m["pi"]))
    for g in sorted(set(reg_s)):
        linhas.append({"modelo": nome, "grupo": "regiao", "nivel": g, "n_mun": int((reg_s == g).sum()), "dp": r_[reg_s == g].std(ddof=1)})
    for g in sorted(set(q_n)):
        linhas.append({"modelo": nome, "grupo": "quintil_n", "nivel": g, "n_mun": int((q_n == g).sum()), "dp": r_[q_n == g].std(ddof=1)})
    linhas.append({"modelo": nome, "grupo": "spearman_abs_r", "nivel": "eta", "n_mun": N, "dp": stats.spearmanr(np.abs(r_), eta_)[0]})
    linhas.append({"modelo": nome, "grupo": "spearman_abs_r", "nivel": "log_n", "n_mun": N, "dp": stats.spearmanr(np.abs(r_), np.log(n))[0]})
VAR_F = pd.DataFrame(linhas).rename(columns={"dp": "valor"}); VAR_F.to_csv(TAB / "diag_final_variancia.csv", index=False)
print(VAR_F.pivot_table(index=["grupo", "nivel"], columns="modelo", values="valor").round(3).to_string())

modelo                       bb_selecao  bb_slx
grupo          nivel                           
quintil_n      Q1                1.0760  1.0370
               Q2                1.0600  1.0140
               Q3                1.0650  1.0280
               Q4                1.0890  1.0410
               Q5                0.8570  0.9950
regiao         Centro-Oeste      0.9510  1.0000
               Nordeste          1.1830  1.0350
               Norte             1.4340  1.0930
               Sudeste           0.8040  1.0090
               Sul               0.9250  1.0120
spearman_abs_r eta               0.0950  0.0040
               log_n            -0.0880 -0.0060


**Leitura.** `bb_selecao`: DP por região de 0,80 (Sudeste) a 1,43 (Norte); por quintil de $n$ de 0,86 (Q5) a 1,09 (Q4); Spearman de $|r|$ com $\hat\eta$ = 0,095 e com $\log n$ = -0,088. `bb_slx`: DP por região de 1,00 (Centro-Oeste) a 1,09 (Norte); por quintil de $n$ de 0,99 (Q5) a 1,04 (Q4); Spearman de $|r|$ com $\hat\eta$ = 0,004 e com $\log n$ = -0,006. 

#### 6.5 · Independência espacial dos resíduos

I de Moran dos resíduos quantílicos (rainha, 999 permutações, `np.random.seed(0)`). A inferência dos coeficientes usa a covariância de Conley (Parte 15).

In [32]:
# Independência: I de Moran (999 permutações, pesos rainha da célula do Passo 5) dos resíduos quantílicos
linhas = []
for nome in MS:
    np.random.seed(0)
    mi = esda.Moran(RF[nome], w, permutations=999)
    linhas.append({"modelo": nome, "I": mi.I, "esperado": mi.EI, "z_sim": mi.z_sim, "p_sim": mi.p_sim})
MORAN_F = pd.DataFrame(linhas); MORAN_F.to_csv(TAB / "diag_final_moran.csv", index=False)
print(MORAN_F.round(4).to_string(index=False))

    modelo      I  esperado   z_sim  p_sim
bb_selecao 0.3270   -0.0002 41.2967 0.0010
    bb_slx 0.3361   -0.0002 41.8656 0.0010


**Leitura.** `bb_selecao`: I de Moran = 0,327 (esperado -0,0002; z simulado 41,3; p simulado = 0,001, o mínimo com 999 permutações). `bb_slx`: I de Moran = 0,336 (esperado -0,0002; z simulado 41,9; p simulado = 0,001, o mínimo com 999 permutações). **Veredito pela regra:** a independência falha nos dois modelos, como em todos os anteriores; o I do `bb_slx` (0,336) não é menor que o do `bb_selecao` (0,327), isto é, o SLX não remove a autocorrelação dos resíduos. A correlação que sobra é tratada na inferência pelo erro-padrão de Conley (Parte 15), não aqui.

#### 6.6 · Colinearidade (VIF/GVIF) do `bb_slx`

Matriz de média padronizada final, com as defasagens `w_*`. Fatores (região, bloco): $\mathrm{GVIF}^{1/(2\,gl)}$, comparado a $\sqrt{10}$. Número de condição de Belsley (colunas com norma 1, com constante).

In [33]:
# Colinearidade: VIF/GVIF da matriz de média final padronizada (inclui as w_*)
if "bb_slx" in MS:
    m = MS["bb_slx"]; Xf = E.montar_X(m["df"], m["mapa"], m["aj"].pad, m["cand"].termos)
    grupos = {"regiao": [c for c in Xf.columns if c.startswith("reg_")], "bloco": [c for c in Xf.columns if c.startswith("bloco_")]}
    VIF = PR.gvif(Xf, grupos).sort_values("gvif_ajustado", ascending=False)
    VIF["acima_10"] = VIF.gvif_ajustado ** 2 > 10   # fator: GVIF^(1/(2gl)) ao quadrado > 10; gl = 1: VIF > 10
    cond = PR.numero_condicao(Xf)
    VIF.to_csv(TAB / "diag_final_vif.csv", index=False)
    pd.DataFrame({"numero_condicao": [cond], "colunas": [Xf.shape[1]]}).to_csv(TAB / "diag_final_condicao.csv", index=False)
    print(VIF.round(2).to_string(index=False)); print(f"número de condição (Belsley, colunas com norma 1, com constante): {cond:.1f}")
    print("fatores acima do limite (VIF > 10; fator: GVIF^(1/(2gl)) > sqrt(10)):", VIF.loc[VIF.acima_10, "fator"].tolist())
else:
    print("bb_slx ausente: VIF pulado.")

              fator  gl     vif  gvif_ajustado  acima_10
w_pct_pretos_pardos   1 21.5500         4.6400      True
              w_lag   1 18.4200         4.2900      True
    w_cobertura_pbf   1 17.1600         4.1400      True
  pct_pretos_pardos   1 17.0300         4.1300      True
        w_pct_16_24   1 12.4800         3.5300      True
                lag   1 12.0300         3.4700      True
         w_log_dens   1  9.7400         3.1200     False
           log_dens   1  8.3700         2.8900     False
          pct_16_24   1  8.0200         2.8300     False
      cobertura_pbf   1  7.7400         2.7800     False
   w_pct_analfabeto   1  7.1900         2.6800     False
     pct_analfabeto   1  5.1500         2.2700     False
     w_pct_superior   1  4.6700         2.1600     False
  w_pct_evangelicos   1  4.4300         2.1000     False
   w_comparecimento   1  4.2600         2.0600     False
     w_pct_mulheres   1  3.8600         1.9600     False
     w_pct_vab_agro   1  3.7700

**Leitura.** Número de condição de Belsley = 19,5. 6 de 26 fatores passam do limite (VIF > 10): `w_pct_pretos_pardos` (21,5), `w_lag` (18,4), `w_cobertura_pbf` (17,2), `pct_pretos_pardos` (17,0), `w_pct_16_24` (12,5), `lag` (12,0). O maior é `w_pct_pretos_pardos` (21,5). Região: GVIF = 24,9 com 3 gl, GVIF^(1/(2gl)) = 1,71 (limite $\sqrt{10}$ = 3,16); bloco: 1,04. As defasagens `w_*` têm VIF médio 9,1 contra 6,3 das covariáveis próprias. **Veredito pela regra:** a leitura de coeficientes individuais é frágil para os fatores listados (cada covariável compartilha informação com a própria defasagem espacial); a previsão não é afetada, e o ganho do SLX foi medido por validação cruzada (Parte 14), não por coeficientes. Remédio possível, não aplicado: ler os pares $(x, Wx)$ conjuntamente por Wald de 2 gl.

#### 6.7 · Influência: DFBETA e Cook de um passo

Pega os 10 maiores Cook de cada modelo, reajusta sem eles e mede a mudança de cada termo em EP de Conley (todos os parâmetros, inclusive os de dispersão).

In [34]:
# Influência beta-binomial: DFBETA de um passo, Cook de um passo, reajuste sem os 10 maiores
linhas, linhas_top = [], []
for nome, m in MS.items():
    res = m["res"]; sc = res.model.score_obs(res.params); Hs = res.model.hessian(res.params)
    d, cook = PR.cook_um_passo(sc, Hs)
    top = np.argsort(-cook)[:10]
    cov_c = np.asarray(S.cov_conley(res, lat, lon, CORTE_KM), float)
    res_sem = PR.reajuste_sem(res, top)
    delta = PR.mudanca_em_ep(res.params, res_sem.params, cov_c)
    termos_ = list(m["aj"].nomes)
    for t_, dl in zip(termos_, delta):
        linhas.append({"modelo": nome, "termo": t_, "beta": float(res.params[termos_.index(t_)]), "mudanca_ep_conley": dl,
                       "troca_sinal": bool(np.sign(res_sem.params[termos_.index(t_)]) != np.sign(res.params[termos_.index(t_)]))})
    for i in top:
        linhas_top.append({"modelo": nome, "municipio": df["nome_municipio"].iloc[i], "uf": df["sigla_uf"].iloc[i], "n": n[i], "cook": cook[i]})
INF = pd.DataFrame(linhas); INF.to_csv(TAB / "diag_final_influencia.csv", index=False)
TOP = pd.DataFrame(linhas_top); TOP.to_csv(TAB / "diag_final_cook_top10.csv", index=False)
INF_RES = INF.groupby("modelo").apply(lambda g: pd.Series({"max_abs_ep": g.mudanca_ep_conley.abs().max(),
    "termo_max": g.loc[g.mudanca_ep_conley.abs().idxmax(), "termo"], "n_gt_05ep": int((g.mudanca_ep_conley.abs() > 0.5).sum()),
    "n_termos": len(g), "trocas_sinal": int(g.troca_sinal.sum())}))
INF_RES.to_csv(TAB / "diag_final_influencia_resumo.csv")
print(TOP.groupby("modelo").head(10).to_string(index=False)); print(INF_RES.to_string())
print("tempo do Passo 6:", round(time.time() - t6), "s")

    modelo             municipio uf              n   cook
bb_selecao              Uiramutã RR     7,596.0000 0.0484
bb_selecao                 Jutaí AM    10,825.0000 0.0235
bb_selecao             Normandia RR     7,359.0000 0.0138
bb_selecao Senador José Porfírio PA    10,721.0000 0.0132
bb_selecao            Belo Monte AL     4,566.0000 0.0124
bb_selecao           Mato Grosso PB     2,223.0000 0.0103
bb_selecao            São Julião PI     4,434.0000 0.0085
bb_selecao   Santa Rosa do Purus AC     2,599.0000 0.0084
bb_selecao           Nova Nazaré MT     2,443.0000 0.0070
bb_selecao                 Anapu PA    18,459.0000 0.0067
    bb_slx                 Jutaí AM    10,825.0000 0.0210
    bb_slx              Brasília DF 1,804,766.0000 0.0155
    bb_slx Senador José Porfírio PA    10,721.0000 0.0129
    bb_slx          Campinápolis MT     8,359.0000 0.0115
    bb_slx           Nova Nazaré MT     2,443.0000 0.0086
    bb_slx                Maricá RJ   143,341.0000 0.0070
    bb_slx    

**Leitura.** `bb_selecao`: os 10 maiores Cook vão de 0,0484 a 0,0067; lideram Uiramutã/RR, Jutaí/AM, Normandia/RR. Sem eles, a maior mudança é 0,73 EP de Conley (`log_theta`), 2 de 19 termos passam de 0,5 EP (`bloco_outro` 0,64, `log_theta` 0,73) e há 0 troca(s) de sinal. `bb_slx`: os 10 maiores Cook vão de 0,0210 a 0,0065; lideram Jutaí/AM, Brasília/DF, Senador José Porfírio/PA. Sem eles, a maior mudança é 0,86 EP de Conley (`disp_reg_Norte`), 2 de 36 termos passam de 0,5 EP (`bloco_outro` 0,60, `disp_reg_Norte` 0,86) e há 0 troca(s) de sinal. **Veredito pela regra:** nenhum termo troca de sinal e o maior deslocamento é abaixo de 1 EP; a variância beta-binomial pesa menos as cidades enormes do que o quase-binomial do Passo 3 (21 de 29 coeficientes acima de 0,5 EP). Termos acima de 0,5 EP estão listados acima.

### Leitura da Parte 10: o que os diagnósticos permitem concluir

Nenhum modelo é escolhido aqui; o notebook 03 decide por validação cruzada. Aqui só dizemos quais pressupostos se sustentam. Números: tabelas `results/tables/diag_*.csv`.

**Distribuição da resposta.**
- `bin_logit`, `quasi_reglag` e `glmm_uf` **falham**: desvio padrão dos resíduos quantílicos 3,45, 3,44 e 3,26 (esperado 1), 52% a 56% dos resíduos fora de ±2 (esperado 4,55%) e 100% do half-normal do `bin_logit` fora do envelope. O quase-binomial corrige o erro-padrão por φ̂ = 21,6, mas não a forma da distribuição; o efeito aleatório de UF reduz a dispersão em apenas ~5%.
- Os três beta-binomiais (`bb_logit`, `bb_reglag`, `bb_spline`) **acertam a escala média** (desvio padrão 1,03; 4,8% a 5,0% fora de ±2) e a escolha de θ, mas **falham no formato**: assimetria -0,42 a -0,45, curtose em excesso 3,8 a 4,0 e 1,06% a 1,11% fora de ±3 (esperado 0,27%). No half-normal de `bb_reglag`, 87% dos pontos ficam abaixo da banda (corpo estreito demais) e 5,2% acima (caudas pesadas demais). O desvio padrão do resíduo varia por região, de 0,80 (Sudeste) a 1,40 (Norte), ou seja, **a dispersão θ não é constante**: o pressuposto de uma precisão única não se sustenta, e a evidência é de heterogeneidade regional da dispersão, não de falta de ajuste na média.

**Preditor linear.** Resíduos de todos os modelos têm correlação de Spearman ≤ 0,035 (em módulo) com o ajustado e com o `lag`, e médias regionais próximas de 0 nos beta-binomiais (|média| ≤ 0,02): não há tendência nem forma funcional visivelmente errada. `bb_spline` e `bb_reglag` quase não melhoram os resíduos frente a `bb_logit`.

**Pontos influentes.** Os 10 maiores valores de Cook são capitais e grandes cidades (Brasília e São Paulo concentram D = 6,0 e 4,1; a alavanca de Brasília é 0,67). No quase-binomial, retirá-los muda 21 dos 29 coeficientes em mais de 0,5 EP, 3 trocam de sinal e o maior deslocamento é 9,0 EP (`bloco_outro`): o quase-binomial **é sensível** a poucas cidades. No `bb_reglag` o maior deslocamento é 0,15 EP e nenhum dos 30 coeficientes passa de 0,5 EP, porque a variância beta-binomial pesa menos os municípios enormes.

**Independência espacial.** Falha em todos: I de Moran dos resíduos de 0,29 a 0,36 (p_sim = 0,001, o mínimo com 999 permutações). Os preditores (e a UF no GLMM, I = 0,29, o menor) explicam parte da estrutura espacial da EDA (I = 0,90 na proporção), mas sobra muita. Consequência: **os erros-padrão e intervalos de todos os modelos continuam pequenos demais** e testes de significância dos coeficientes não devem ser lidos de forma literal. A estimativa pontual de π não é necessariamente enviesada; o desempenho preditivo honesto sai da validação deixando uma UF de fora (notebook 03).

**Resumo por modelo.**

| modelo | distribuição | preditor | influentes | independência |
|---|---|---|---|---|
| `bin_logit` | falha | ok | não avaliado (usa-se o quase) | falha |
| `quasi_reglag` | falha na forma (EP corrigido por φ) | ok | falha | falha |
| `bb_logit`, `bb_spline` | escala ok, formato e θ constante falham | ok | ok (modelo irmão `bb_reglag`) | falha |
| `bb_reglag` | escala ok, formato e θ constante falham | ok | ok | falha |
| `glmm_uf` | falha | ok | não avaliado | falha (menor I) |
| `bb_selecao` (Passo 6) | escala ok, formato falha | ligação sinalizada, efeito pequeno; `pct_pretos_pardos` não linear | ok (máx. 0,7 EP, sem troca de sinal) | falha |
| `bb_slx` (Passo 6) | escala ok por região, formato falha (menos) | ligação sinalizada, efeito pequeno; `pct_pretos_pardos` não linear | ok (máx. 0,9 EP, sem troca de sinal) | falha; VIF alto nos pares x, Wx |

Remédios: a dispersão por região (θ função da região) e o SLX foram testados e aceitos no notebook 05 (Partes 13 e 14); ESF, krigagem e CAR foram rejeitados (Partes 14 e 17); a inferência usa erros-padrão de Conley (Parte 15).

**Modelos escolhidos (Passo 6).** `bb_selecao` e `bb_slx` passaram pela bateria completa (ligação, linearidade, distribuição, variância, independência, colinearidade e influência). A dispersão por região acerta a escala mas não o formato da distribuição; a ligação logit é sinalizada pelo teste com efeito pequeno nas previsões; só `pct_pretos_pardos` mostra curvatura material; a independência segue falhando (Conley cobre a inferência); os pares $(x, Wx)$ são colineares, o que fragiliza coeficientes individuais mas não a previsão; nenhum município muda o sinal de um coeficiente. Remédios candidatos, sem aplicar: spline em `pct_pretos_pardos`, família de cauda mais pesada e leitura conjunta dos pares $(x, Wx)$.
